<a id="build-your-own-hybrid-rag-system-from-scratch"></a>
# Build Your Own Hybrid RAG System — From Scratch

**AI4Metascience School — CNRS AISSAI Center**
*Domaine Saint-Paul, Saint-Rémy-lès-Chevreuse — Sept/Oct 2026*

---

<a id="what-you-will-build-in-this-notebook"></a>
## What you will build in this notebook

By the end of this notebook you will have built, **entirely from scratch and entirely locally**,
a hybrid Retrieval-Augmented Generation (RAG) pipeline over a real technical documentation corpus
(scraped from a French national computing center, [cc.in2p3.fr](https://cc.in2p3.fr/)):

1. **Setup & environment check** — make sure everything runs on a CPU-only laptop with 16 GB RAM.
2. **Corpus acquisition** — politely scrape technical documentation pages.
3. **Chunking strategies** — split long documents into retrievable units.
4. **Embeddings (dense + sparse)** — encode chunks with a single multilingual model (BGE-M3).
5. **Hybrid vector store** — store dense vectors in ChromaDB, sparse (lexical) weights alongside.
6. **Hybrid retrieval + Reciprocal Rank Fusion (RRF)** — combine dense and sparse search.
7. **Conversational scenario + local LLM generation** — use the top similarity score to decide
   how to answer, then generate a final answer with a small local LLM served by Ollama.

> **Why "from scratch"?** Frameworks like LangChain or LlamaIndex hide almost every interesting
> decision behind a one-liner. Here, every step is visible and editable: you will see exactly
> what a chunk looks like, what an embedding vector looks like, how a sparse vector is represented,
> and how two ranked lists get fused into one. This is what makes the difference between
> "I used a RAG library" and "I understand what a RAG system does."

<a id="target-hardware-for-this-main-track"></a>
## Target hardware for this main track

| Component | Minimum | Comfortable |
|---|---|---|
| RAM | 16 GB | 32 GB |
| CPU | 4 cores | 8 cores |
| GPU | none required | optional, speeds up embeddings/LLM |
| Disk | 8 GB free | 15 GB free |
| OS | Linux / macOS / Windows + WSL2 | same |

Everything below runs **CPU-only**. If your laptop has a GPU, the same code will automatically
use it (PyTorch and Ollama both auto-detect CUDA/Metal), but nothing here *requires* one.

<a id="estimated-time-budget"></a>
## Estimated time budget

| Section | Time |
|---|---|
| 1. Setup | 20 min |
| 2. Corpus acquisition | 35 min |
| 3. Chunking | 35 min |
| 4. Embeddings (dense + sparse) | 50 min |
| 5. Hybrid vector store | 40 min |
| 6. Hybrid retrieval + RRF | 45 min |
| 7. Conversational scenario + LLM generation | 55 min |
| **Total** | **~4h20** (leaves room for questions / debugging in a 6h slot) |

---


<p align="center">
  <img src="https://media.geeksforgeeks.org/wp-content/uploads/20250210190608027719/How-Rag-works.webp" alt="Retrieval-Augmented Generation pipeline" width="620"><br>
  <sub><i>Vue d'ensemble du pipeline RAG que ce notebook construit pas à pas — [GeeksforGeeks](https://www.geeksforgeeks.org/nlp/what-is-retrieval-augmented-generation-rag/)</i></sub>
</p>


## 🎓 About this course — LaboBots School

**Target audience**: researchers, engineers, and IT staff (DSI) from French public research
institutions (CEA, CNRS, universities, and similar organizations) wanting to understand and
deploy, end to end, a Retrieval-Augmented Generation (RAG) system adapted to a lab or research
center context.

**Course goal**: progressively build and deploy the same hybrid RAG pipeline (dense search +
lexical search, fused via Reciprocal Rank Fusion), exploring several architectures — from a
fully local execution up to distributed architectures representative of what you'd actually
encounter in a lab: shared GPU compute resources, institutional computing center infrastructure,
token-based authentication, etc.

The course is structured across three notebooks, each one shifting the slider on "what stays in
the lab, and what gets offloaded?":

| Notebook | Retriever (vector store) | Generation (LLM) | Web interface | Authentication |
|---|---|---|---|---|
| **1. From scratch (local)** | Local — embedded ChromaDB | Local — Ollama | Jupyter notebook (laptop) | None |
| **2. Distributed architecture (GPU)** | Remote — dedicated GPU compute machine | Remote — same GPU machine (Ollama + LiteLLM proxy, multi-user) | Local — Streamlit (laptop) | Per-participant LiteLLM API key, SSH tunnel |
| **3. Computing center infrastructure (coming soon)** | Remote — Computing Center (CC) machine | Remote — LLM models hosted by the CC | Local — Streamlit (laptop) | CC access token |

This slider can be set differently depending on your own lab's constraints: for example, you
could **keep all of the retriever's data in the lab** (data sovereignty) and only use remote
compute resources for generation, or conversely **host everything remotely** and keep only the
web interface in the lab — this is exactly what this course illustrates, notebook after notebook.

**You are here: Notebook 1 — entirely local, CPU only.**
No remote resource is used: the crawler, chunking, embeddings (dense and lexical, via BGE-M3),
the vector store (embedded ChromaDB), and the generation LLM (Ollama) all run on your laptop, no
GPU required. This is the foundational understanding that notebooks 2 and 3 build on.

## 🗂️ Plan de ce notebook

- [What you will build in this notebook](#what-you-will-build-in-this-notebook)
- [Target hardware for this main track](#target-hardware-for-this-main-track)
- [Estimated time budget](#estimated-time-budget)
- [1. Setup & environment check](#sec-1)
  - [1.0 A 60-second primer: what is an LLM, and what is Ollama?](#sec-1-0)
  - [1.1 What we need and why](#sec-1-1)
  - [1.2 Install Ollama (one-time, outside this notebook)](#sec-1-2)
  - [1.3 Python dependencies](#sec-1-3)
- [2. Corpus acquisition — scraping cc.in2p3.fr](#sec-2)
  - [2.1 Why this corpus?](#sec-2-1)
  - [2.2 Netiquette first — please read before running](#sec-2-2)
  - [2.3 Getting *complete* subtree coverage, not just what link-following happens to find](#sec-2-3)
  - [2.4 Loading the full pre-scraped corpus](#sec-2-4)
- [3. Chunking strategies](#sec-3)
  - [3.1 Why chunk at all — and why NOT just index whole pages?](#sec-3-1)
  - [3.2 A structural fix upstream: chunk within headings, not across them](#sec-3-2)
  - [3.3 Chunk metadata](#sec-3-3)
  - [3.4 Heading-aware chunking + contextual headers](#sec-3-4)
  - [3.5 Build the full chunk list](#sec-3-5)
  - [3.6 (Optional, advanced) LLM-generated contextual summaries instead of a structural breadcrumb](#sec-3-6)
- [4. Embeddings — dense *and* sparse, from a single model (BGE-M3)](#sec-4)
  - [4.1 Why BGE-M3?](#sec-4-1)
  - [4.2 Model size / performance note for CPU-only laptops](#sec-4-2)
  - [4.3 Encode the full corpus](#sec-4-3)
  - [4.4 Fallback: classic BM25 sparse index (if you're not using BGE-M3's native sparse output)](#sec-4-4)
- [5. Building the hybrid vector store](#sec-5)
  - [5.1 Why ChromaDB?](#sec-5-1)
  - [5.2 Create a persistent Chroma collection](#sec-5-2)
  - [5.3 Populate the collection](#sec-5-3)
- [6. Hybrid retrieval — combining dense and sparse search with Reciprocal Rank Fusion](#sec-6)
  - [6.1 The problem with combining raw scores](#sec-6-1)
  - [6.2 Reciprocal Rank Fusion (RRF)](#sec-6-2)
  - [6.3 Implementing sparse (lexical) scoring manually](#sec-6-3)
  - [6.4 Compare: dense-only vs sparse-only vs hybrid](#sec-6-4)
- [7. Conversational scenario driven by similarity, and generation with a local LLM](#sec-7)
  - [7.1 Why gate the answer on the similarity score?](#sec-7-1)
  - [7.2 The role of the LLM here](#sec-7-2)
  - [7.3 "Small-to-big": search with chunks, generate with more context](#sec-7-3)
  - [7.4 Try it yourself](#sec-7-4)
  - [7.5 Calibrating the band boundaries properly (going further)](#sec-7-5)
  - [7.6 (Bonus) Comparing seven retrieval strategies side by side](#sec-7-6)
  - [7.6.1 Building the three new indexes](#sec-7-6-1)
  - [7.6.2 Search functions for the three new strategies](#sec-7-6-2)
  - [7.6.3 Evaluation harness](#sec-7-6-3)
  - [7.6.4 Reading the table](#sec-7-6-4)
  - [7.6.5 Push the cached summaries into their own Chroma collection](#sec-7-6-5)
- [Checkpoint](#checkpoint)
- [🚀 Final project: bring your own documentation](#final-project)


<a id="sec-1"></a>
## 1. Setup & environment check

<a id="sec-1-0"></a>
### 1.0 A 60-second primer: what is an LLM, and what is Ollama?

**LLM (Large Language Model)**: a neural network -- specifically a *Transformer* (see the links
below) -- trained on huge amounts of text to predict "what word/token comes next," applied
repeatedly to generate whole answers one token at a time. Its size is usually described by its
**parameter count** (e.g. "3B" = 3 billion weights): roughly, more parameters means more capacity
to capture patterns, at the cost of more memory and compute per query. `llama3.2:3b` (installed in
1.2 below) *is* the neural network doing the "understand language, generate text" work in this
notebook.

**Ollama**: *not* an LLM itself -- it's the **runtime/server** that downloads, quantizes (shrinks a
model to fit in less RAM, at a small quality cost), loads into memory, and serves an LLM over a
simple local HTTP API (`/api/chat`, used by the `ollama` Python client in Section 7). Think of it
as similar to what Docker is for containers: Ollama doesn't *contain* the intelligence, it runs and
exposes whatever model you point it at -- swapping `llama3.2:3b` for a different model in 1.2
changes the LLM; Ollama itself stays exactly the same.

**Want the mechanism behind "neural network trained to predict the next token"?**
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) (Jay
  Alammar) -- the standard, accessible visual walkthrough of the architecture behind every modern
  LLM, and behind BGE-M3's embeddings in Section 4.
- *[Attention Is All You Need](https://arxiv.org/abs/1706.03762)* (Vaswani et al., 2017) -- the
  original paper; Figure 1 is the canonical Transformer diagram, denser to read but the source.
- [ollama.com/library](https://ollama.com/library) -- browse other models Ollama can serve, with
  their parameter counts and quantization options (the same page for `llama3.2:3b` shows exactly
  what you're pulling in 1.2).

A related but distinct concept -- **embedding** vectors (dense *and* sparse) -- comes back in
Section 4, once there's a real corpus to embed.

<a id="sec-1-1"></a>
### 1.1 What we need and why

| Tool | Role in the pipeline | Why this one |
|---|---|---|
| `requests` + `beautifulsoup4` | Fetch & parse HTML documentation pages | Standard, lightweight, no JS engine needed for static doc sites |
| `FlagEmbedding` (BGE-M3) | Produce **dense** and **sparse** embeddings from a single model | One model, two representations — ideal for hybrid search without juggling two libraries |
| `chromadb` | Store & query dense vectors | Embedded (no server to run), simple API, perfect for a from-scratch pedagogy |
| `numpy` | Manual scoring, cosine similarity, RRF fusion | We want to *see* the math, not hide it behind a framework |
| `ollama` (external app) + `ollama` Python client | Serve a local LLM for the final generation step | Free, open, runs fully offline, simple CLI to pull models |

<a id="sec-1-2"></a>
### 1.2 Install Ollama (one-time, outside this notebook)

Ollama is **not** a Python package — it's a small background service. If you haven't installed it yet:

```bash
# Linux
curl -fsSL https://ollama.com/install.sh | sh

# macOS
# download from https://ollama.com/download

# Windows
# download the installer from https://ollama.com/download
```

Then pull the small model we'll use for generation (about 2 GB download):

```bash
ollama pull llama3.2:3b
```

> **Bandwidth warning for the school**: if 30 people pull this at the same time on the venue's
> Wi-Fi, it will be painful.

<a id="sec-1-3"></a>
### 1.3 Python dependencies

Run the cell below once. It installs everything needed for sections 1–7.

<p align="center">
  <img src="https://raw.githubusercontent.com/ollama/ollama/main/docs/ollama-logo.svg" alt="Ollama logo" width="90"><br>
  <sub><i>Ollama — sert le LLM local utilisé pour la génération (Section 7)</i></sub>
</p>


In [1]:
# Run this once. On a fresh laptop this can take a few minutes (BGE-M3 dependencies are not tiny).
# If you are offline during the session, this must have been run BEFORE arriving.

%pip install -q requests beautifulsoup4 lxml tqdm numpy scikit-learn \
               chromadb FlagEmbedding ollama
print("Dependencies installed (or already present).")

Note: you may need to restart the kernel to use updated packages.
Dependencies installed (or already present).


In [2]:
# --- Sanity check: import everything we will use, and check versions ---
import sys, platform, importlib

packages = ["requests", "bs4", "numpy", "sklearn", "chromadb", "FlagEmbedding", "ollama"]
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print("-" * 60)
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown version")
        print(f"OK   {pkg:<15} {version}")
    except ImportError as e:
        print(f"FAIL {pkg:<15} -> {e}")

Python: 3.13.0 | packaged by Anaconda, Inc. | (main, Oct  7 2024, 21:29:38) [GCC 11.2.0]
Platform: Linux-6.12.107+deb13-amd64-x86_64-with-glibc2.41
------------------------------------------------------------
OK   requests        2.32.5
OK   bs4             4.15.0
OK   numpy           2.4.6
OK   sklearn         1.8.0
OK   chromadb        1.5.2
OK   FlagEmbedding   unknown version
OK   ollama          unknown version


In [3]:
# --- Check that Ollama is reachable and the model is pulled ---
import ollama

MODEL_NAME = "llama3.2:3b"

try:
    models = ollama.list()
    names = [m["model"] for m in models.get("models", [])]
    print("Ollama is running. Available models:", names)
    if not any(MODEL_NAME in n for n in names):
        print(f"\n⚠️  '{MODEL_NAME}' not found locally. Run in a terminal:\n    ollama pull {MODEL_NAME}")
    else:
        print(f"\n✅ '{MODEL_NAME}' is ready to use.")
except Exception as e:
    print("⚠️  Could not reach Ollama. Is the Ollama app/service running?")
    print("Error:", e)

Ollama is running. Available models: ['llama3.2:3b', 'gemma4:e4b']

✅ 'llama3.2:3b' is ready to use.


In [4]:
# --- Working directories for this notebook ---
import os

WORKDIR = os.path.abspath("./rag_workshop")
CORPUS_DIR = os.path.join(WORKDIR, "corpus")
CHROMA_DIR = os.path.join(WORKDIR, "chroma_db")

for d in (WORKDIR, CORPUS_DIR, CHROMA_DIR):
    os.makedirs(d, exist_ok=True)

print("Working directory:", WORKDIR)
print("Corpus directory :", CORPUS_DIR)
print("Chroma directory :", CHROMA_DIR)

Working directory: /home/anne/Documents/LaboBots/rag_workshop
Corpus directory : /home/anne/Documents/LaboBots/rag_workshop/corpus
Chroma directory : /home/anne/Documents/LaboBots/rag_workshop/chroma_db


<a id="sec-2"></a>
## 2. Corpus acquisition — scraping cc.in2p3.fr

<a id="sec-2-1"></a>
### 2.1 Why this corpus?

[cc.in2p3.fr](https://cc.in2p3.fr/) is the website of the **IN2P3 Computing Center** (CNRS),
a real French national HPC/data center. Its documentation is technical, French/English mixed,
and structurally similar to what you will find in your own lab's wiki or documentation site —
which is exactly the kind of corpus your final RAG system needs to handle. This makes it a
realistic training ground before the final exercise, where you'll point this same pipeline at
**your own lab's documentation**.

<a id="sec-2-2"></a>
### 2.2 Netiquette first — please read before running

We are about to write a small web crawler. A few rules we **always** follow, and that you should
carry over to your own lab's scraping later:

1. **Check `robots.txt`** and respect `Disallow` rules.
2. **Rate-limit requests** (a delay between requests — never hammer a server).
3. **Set a descriptive User-Agent** so the site owner can identify and, if needed, contact you.
4. **Limit scope** — crawl a bounded number of pages, not the entire internet reachable from the seed URL.
5. **Cache what you fetch** — don't re-download the same page twice across cells/re-runs.

> ⚠️ **During the live session**: if 30 participants crawl the same live site at the same time,
> we risk overloading it or getting temporarily blocked. The instructor will hand out a
> **pre-scraped corpus** (`corpus_ccin2p3.json`) for everyone to use in sections 3–7. The crawler
> below is here so you understand *how* that corpus was built, and you are welcome to re-run it
> at home with a larger page budget. During the workshop, run it with the small `MAX_PAGES`
> below (10–15 pages) just to see it work, then switch to the pre-scraped file.

<a id="sec-2-3"></a>
### 2.3 Getting *complete* subtree coverage, not just what link-following happens to find

A naive link-following crawler has three coverage gaps that matter a lot for a documentation
site, where you specifically want the *whole* subtree under, say, `/docs/` or `/wiki/`:

1. **Links hidden in navigation menus get missed if you strip them before harvesting.**
   Documentation sites usually put their *complete* table of contents in a sidebar `<nav>` —
   which is exactly the element you'd be tempted to strip out when extracting clean body text.
   If you strip-then-harvest, you silently lose the main map of the site. **We harvest links
   first, from the untouched page, and only strip navigation afterwards for text extraction.**
2. **A single seed URL plus "follow every `<a>` tag" can wander outside the subtree you actually
   want** (into a blog, a shop, an unrelated section of the same domain), while also missing
   orphan pages that exist but aren't linked from anywhere you crawled. **A `sitemap.xml` (most
   documentation platforms publish one) is a much more reliable source of the complete page
   list** than following links — we check for one and use it as a primary URL source, with
   link-following as a complement that also catches pages not yet listed in the sitemap.
3. **Duplicate URLs that differ only by trailing slash, query string, or case** waste crawl
   budget and can make you think you've covered more than you have. **We normalize URLs before
   deduplicating.**

The crawler below addresses all three. If you only take one thing from this section: *always
harvest links from the raw page, and always check for a sitemap before trusting link-following
alone.*

In [ ]:
import time
import urllib.robotparser as robotparser
from urllib.parse import urljoin, urlparse, urlunparse, parse_qsl, urlencode
import requests
from bs4 import BeautifulSoup

SEED_URL = "https://doc.cc.in2p3.fr/"   # documentation entry point
ALLOWED_DOMAIN = "doc.cc.in2p3.fr"
ALLOWED_PATH_PREFIX = None               # e.g. "/docs/" to scope the crawl to a subtree; None = whole domain
USER_AGENT = "AISSAI-RAG-School-Bot/1.0 (educational crawl; contact: your-email@example.org)"
REQUEST_DELAY_SECONDS = 1.0             # be polite

# --- Pick ONE, comment out the other -------------------------------------------------------
MAX_PAGES = 15          # Quick local iteration: fast (~15s), small corpus -- good while you're
                         # still working through this notebook's cells and don't want to wait.
#MAX_PAGES = None        # Full crawl: every reachable page, no cap (doc.cc.in2p3.fr has ~110-150
                         # of them) -- use this for the real workshop demo, where a complete,
                         # accurate knowledge base is the whole point. Still polite (one request
                         # per REQUEST_DELAY_SECONDS), so budget a few minutes, not seconds.

# Query-string keys that are pure noise for crawling purposes (tracking, sorting, etc.) --
# dropping them avoids treating "/page?ref=nav" and "/page?ref=footer" as two different pages.
IGNORED_QUERY_KEYS = {"utm_source", "utm_medium", "utm_campaign", "ref", "fbclid"}

# File extensions that are never worth fetching as "documentation text" -- following a link to
# one of these downloads the whole binary (this site links multi-hundred-MB software archives
# under /_downloads/) and then tries to parse it as HTML, corrupting the corpus with garbage.
SKIPPED_EXTENSIONS = (
    ".zip", ".tar", ".tar.gz", ".tgz", ".gz", ".rar", ".7z",
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".png", ".jpg", ".jpeg", ".gif", ".svg", ".ico",
    ".mp4", ".mp3", ".zip", ".exe", ".dmg", ".iso", ".whl",
)


def is_probably_binary(url: str) -> bool:
    path = urlparse(url).path.lower()
    return path.endswith(SKIPPED_EXTENSIONS)

HEADERS = {"User-Agent": USER_AGENT}

def can_fetch(url: str) -> bool:
    '''Check robots.txt before fetching a URL.'''
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = robotparser.RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(USER_AGENT, url)
    except Exception:
        # If robots.txt is unreachable, default to being cautious but not blocking the workshop.
        return True

print("robots.txt allows crawling seed URL:", can_fetch(SEED_URL))

In [6]:
def normalize_url(url: str) -> str:
    '''
    Canonicalize a URL so equivalent pages don't get crawled twice:
      - drop the fragment (#section-anchor)
      - drop known-noisy query params, sort the remaining ones for a stable order
      - strip a trailing slash (except for the bare domain root)
    '''
    parsed = urlparse(url)
    query_pairs = [(k, v) for k, v in parse_qsl(parsed.query) if k not in IGNORED_QUERY_KEYS]
    query_pairs.sort()
    path = parsed.path.rstrip("/") or "/"
    return urlunparse((parsed.scheme, parsed.netloc, path, "", urlencode(query_pairs), ""))


def in_scope(url: str, allowed_domain: str, allowed_path_prefix: str | None) -> bool:
    parsed = urlparse(url)
    if parsed.netloc != allowed_domain:
        return False
    if allowed_path_prefix and not parsed.path.startswith(allowed_path_prefix):
        return False
    return True


def discover_sitemap_urls(seed_url: str, allowed_domain: str, allowed_path_prefix: str | None) -> list[str]:
    '''
    Look for a sitemap (declared in robots.txt, or at the conventional /sitemap.xml path) and
    return every in-scope URL it lists. This is usually a MUCH more complete source of "every
    page in the subtree" than following links, since it doesn't depend on every page being
    reachable via a visible link from somewhere we happened to crawl.
    '''
    parsed_seed = urlparse(seed_url)
    origin = f"{parsed_seed.scheme}://{parsed_seed.netloc}"
    candidate_sitemap_urls = [f"{origin}/sitemap.xml"]

    try:
        robots_resp = requests.get(f"{origin}/robots.txt", headers=HEADERS, timeout=10)
        if robots_resp.ok:
            for line in robots_resp.text.splitlines():
                if line.lower().startswith("sitemap:"):
                    candidate_sitemap_urls.insert(0, line.split(":", 1)[1].strip())
    except Exception:
        pass

    found_urls = []
    for sitemap_url in candidate_sitemap_urls:
        try:
            resp = requests.get(sitemap_url, headers=HEADERS, timeout=10)
            if not resp.ok or "xml" not in resp.headers.get("Content-Type", "") and "<urlset" not in resp.text[:200]:
                continue
            soup = BeautifulSoup(resp.text, "xml")
            locs = [loc.get_text(strip=True) for loc in soup.find_all("loc")]
            found_urls = [u for u in locs if in_scope(u, allowed_domain, allowed_path_prefix)]
            if found_urls:
                print(f"Found sitemap at {sitemap_url}: {len(found_urls)} in-scope URLs.")
                break
        except Exception:
            continue

    if not found_urls:
        print("No usable sitemap found -- will rely on link-following only (see 2.3 for the caveats).")
    return found_urls

In [ ]:
def crawl(seed_urls: list[str], allowed_domain: str, max_pages: int, delay: float,
          allowed_path_prefix: str | None = None):
    '''
    Breadth-first crawler limited to one domain (optionally one path prefix), seeded with both
    a sitemap (if found) and the given seed URL(s). Returns a list of dicts:
      {"url": ..., "title": ..., "elements": [{"tag": "h1"/"p"/"li"/..., "text": ...}, ...],
       "text": <flattened text, kept for backward compatibility>}

    The "elements" field preserves heading structure and element order -- Section 3 uses it to
    build chunks that never cross a heading boundary and that carry a breadcrumb back to their
    section. Storing only flattened "text" (as a v1 of this notebook did) throws that structure
    away permanently, which is exactly the context loss you're asking about.
    '''
    visited = set()
    queue = list(dict.fromkeys(normalize_url(u) for u in seed_urls))  # dedupe, keep order
    pages = []

    while queue and (max_pages is None or len(pages) < max_pages):
        url = queue.pop(0)
        if url in visited:
            continue
        visited.add(url)

        if is_probably_binary(url):
            print(f"skip (binary/download link): {url}")
            continue

        if not can_fetch(url):
            print(f"skip (robots.txt): {url}")
            continue

        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            resp.raise_for_status()
        except Exception as e:
            print(f"skip (error): {url} -> {e}")
            continue

        soup = BeautifulSoup(resp.text, "lxml")

        # --- Harvest links FIRST, from the untouched page -----------------------------------
        # This is the fix for the coverage bug: sidebar/nav links are usually the main source
        # of a documentation site's full table of contents. Stripping <nav> before looking for
        # links (as a naive version of this crawler might) silently drops most of the subtree.
        for a in soup.find_all("a", href=True):
            next_url = normalize_url(urljoin(url, a["href"]))
            if (in_scope(next_url, allowed_domain, allowed_path_prefix)
                    and next_url not in visited and not is_probably_binary(next_url)):
                queue.append(next_url)

        # --- THEN strip boilerplate and extract structured text for this page ---------------
        for tag in soup(["script", "style", "footer", "header"]):
            tag.decompose()
        # We keep <nav> out of the text extraction (it's usually just links, not prose) but we
        # already harvested its links above -- order matters here.
        for tag in soup.find_all("nav"):
            tag.decompose()

        title = soup.title.get_text(strip=True) if soup.title else url
        elements = [
            {"tag": t.name, "text": t.get_text(" ", strip=True)}
            for t in soup.find_all(["h1", "h2", "h3", "p", "li", "pre", "code"])
        ]
        elements = [e for e in elements if e["text"]]
        flat_text = " ".join(e["text"] for e in elements)

        if len(flat_text.split()) > 30:  # skip near-empty pages
            pages.append({"url": url, "title": title, "elements": elements, "text": flat_text})
            cap_str = str(max_pages) if max_pages is not None else "?"
            print(f"[{len(pages)}/{cap_str}] fetched: {url}  ({len(flat_text.split())} words, {len(elements)} elements)")

        time.sleep(delay)

    return pages

sitemap_urls = discover_sitemap_urls(SEED_URL, ALLOWED_DOMAIN, ALLOWED_PATH_PREFIX)
seed_urls = sitemap_urls if sitemap_urls else [SEED_URL]

live_pages = crawl(seed_urls, ALLOWED_DOMAIN, MAX_PAGES, REQUEST_DELAY_SECONDS, ALLOWED_PATH_PREFIX)
print(f"\nCrawled {len(live_pages)} pages.")

In [8]:
import json

# Save what we just crawled live (small sample)
live_corpus_path = os.path.join(CORPUS_DIR, "corpus_live_sample.json")
with open(live_corpus_path, "w", encoding="utf-8") as f:
    json.dump(live_pages, f, ensure_ascii=False, indent=2)
print("Saved live sample to:", live_corpus_path)

Saved live sample to: /home/anne/Documents/LaboBots/rag_workshop/corpus/corpus_live_sample.json


<a id="sec-2-4"></a>
### 2.4 Loading the full pre-scraped corpus

For the rest of this notebook (sections 3–7), we use the **pre-scraped, larger corpus** provided
by the instructor, so that everyone works with the same data and we don't hit the live site
30 times in parallel. Ask the instructor for `corpus_ccin2p3.json` and place it in
`./rag_workshop/corpus/`. If it's missing, we fall back to the small live sample you just crawled
(the pipeline works exactly the same either way — just with less data).

In [9]:
FULL_CORPUS_PATH = os.path.join(CORPUS_DIR, "corpus_ccin2p3.json")

if os.path.exists(FULL_CORPUS_PATH):
    with open(FULL_CORPUS_PATH, "r", encoding="utf-8") as f:
        raw_pages = json.load(f)
    print(f"Loaded pre-scraped corpus: {len(raw_pages)} pages.")
else:
    print("Pre-scraped corpus not found — falling back to the small live sample from this session.")
    raw_pages = live_pages

# Quick look at one document
if raw_pages:
    sample = raw_pages[0]
    print("\n--- Sample document ---")
    print("URL  :", sample["url"])
    print("Title:", sample["title"])
    print("Text :", sample["text"][:400], "...")

Pre-scraped corpus not found — falling back to the small live sample from this session.

--- Sample document ---
URL  : https://doc.cc.in2p3.fr/
Title: DOCUMENTATION UTILISATEUR — CC-IN2P3
Text : DOCUMENTATION UTILISATEUR Contactez le support Surveillez votre activité Créez et gérez votre compte DOCUMENTATION UTILISATEUR  Le Centre de Calcul de l’IN2P3 (ou CC-IN2P3) est une Unité d’Appui à la Recherche du CNRS (UAR6402) rattachée à l’IN2P3. Cet institut développe et coordonne les recherches en physique des particules, physique nucléaire et physique des astroparticules. Infrastructure de r ...


<a id="sec-3"></a>
## 3. Chunking strategies

<a id="sec-3-1"></a>
### 3.1 Why chunk at all — and why NOT just index whole pages?

Embedding models and LLM context windows are limited. A whole documentation page might be
2,000+ words — too coarse for precise retrieval (the whole page becomes "one topic" even if it
covers five different ones), and possibly too long to embed well. We split each page into
smaller, semantically coherent **chunks**, each of which becomes a candidate for retrieval.

**"Wouldn't keeping the whole page give better results, since we're losing context by
chunking?"** This is a real trade-off, worth being precise about, because the honest answer is
"it depends which part of the pipeline you mean":

| | Small chunks | Whole pages |
|---|---|---|
| **As the *retrieval/search* unit** | ✅ A dense vector represents one focused idea, so similarity scores are sharp. A hybrid RRF fusion works well because ranks are meaningful. | ❌ A page covering 5 sub-topics gets ONE embedding that's an average of all 5 — this is the classic "embedding dilution" problem. A query about sub-topic #3 competes against noise from #1, #2, #4, #5 baked into the same vector. Precision drops, and this is very likely what caused the "hybrid falls apart" symptom you saw. |
| **As the *generation* context (what the LLM actually reads)** | ❌ A tiny, isolated chunk can be ambiguous on its own — "the default is 3" means nothing without knowing *which* setting. | ✅ More surrounding context helps the LLM write a complete, correct answer. |

The resolution is **not** "pick one" — it's to decouple the two roles: search with small,
precise units, but let the *generation* step see more context than the *retrieval* step scored
on. Section 3.4 below and Section 7 implement exactly this ("small-to-big" retrieval), which
should fix the context-loss problem without reintroducing the dilution problem.

There's also a second, cheaper trick: even a small chunk stops being "context-poor" if you tell
it, and the embedding model, *what page and section it came from* before embedding it — covered
in 3.3 and 3.4.

<a id="sec-3-2"></a>
### 3.2 A structural fix upstream: chunk within headings, not across them

The Section 2 crawler now keeps each page's content as an ordered list of tagged elements
(`h1`, `h2`, `h3`, `p`, `li`, `pre`) instead of one flattened string. This lets us chunk **within**
a heading's scope rather than sliding a fixed-size window over the whole page regardless of
structure — so a chunk never silently splices together the end of one section and the start of
an unrelated one, and every chunk can carry a **breadcrumb** back to its heading (e.g.
`"SLURM Job Submission > Job Arrays"`).

We still show the naive fixed-size / flat structure-aware chunkers first (3.3), because seeing
their failure mode on real data is the best way to understand *why* the heading-aware version
(3.4) exists — don't skip straight past the naive versions.

<a id="sec-3-3"></a>
### 3.3 Chunk metadata

Every chunk must keep a pointer back to its source (URL, title, position) — this is what lets
your final chatbot **cite its sources**, which matters a lot for a technical support use case.

In [ ]:
import re
import sys
from typing import List

# Chunk lives in rag_workshop/chunk_types.py (not defined here) so that pickle can resolve
# it by a real module path -- a class defined in a notebook cell is recorded under
# `__main__`, which points to the kernel's own namespace, not to whatever script (e.g. the
# Streamlit app in notebook 2) later tries to unpickle chunks.pkl.
sys.path.insert(0, WORKDIR)
from chunk_types import Chunk

def word_count(text: str) -> int:
    return len(text.split())

def fixed_size_chunk(text: str, max_words: int = 180, overlap_words: int = 30) -> List[str]:
    '''
    Simple sliding-window chunker over whitespace-tokenized words.
    `overlap_words` repeats the tail of a chunk at the start of the next one,
    so we don't lose meaning right at a cut point.
    '''
    words = text.split()
    if len(words) <= max_words:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap_words  # step back for overlap
    return chunks

# Try it on our sample document
sample_text = raw_pages[0]["text"]
demo_chunks = fixed_size_chunk(sample_text, max_words=180, overlap_words=30)
print(f"Document has {word_count(sample_text)} words -> split into {len(demo_chunks)} fixed-size chunks.")
print("\n--- Chunk 0 ---\n", demo_chunks[0][:300], "...")
if len(demo_chunks) > 1:
    print("\n--- Chunk 1 (notice the overlap with the end of chunk 0) ---\n", demo_chunks[1][:300], "...")

In [11]:
def structure_aware_chunk(text: str, max_words: int = 180, overlap_words: int = 30) -> List[str]:
    '''
    A slightly smarter chunker: first split on sentence boundaries, then greedily
    pack sentences into chunks up to max_words, so we never cut a sentence in half.
    '''
    # naive sentence splitter (good enough for documentation prose; a real system
    # might use a proper NLP sentence tokenizer here)
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks, current, current_len = [], [], 0
    for sent in sentences:
        sent_len = word_count(sent)
        if current_len + sent_len > max_words and current:
            chunks.append(" ".join(current))
            # start next chunk with overlap: keep last few words of the previous chunk
            overlap_text = " ".join(" ".join(current).split()[-overlap_words:])
            current = [overlap_text, sent]
            current_len = word_count(overlap_text) + sent_len
        else:
            current.append(sent)
            current_len += sent_len
    if current:
        chunks.append(" ".join(current))
    return chunks

demo_chunks_v2 = structure_aware_chunk(sample_text, max_words=180, overlap_words=30)
print(f"Structure-aware chunking -> {len(demo_chunks_v2)} chunks (vs {len(demo_chunks)} for fixed-size).")
print("\n--- Chunk 0 (structure-aware) ---\n", demo_chunks_v2[0][:300], "...")

Structure-aware chunking -> 1 chunks (vs 1 for fixed-size).

--- Chunk 0 (structure-aware) ---
 DOCUMENTATION UTILISATEUR Contactez le support Surveillez votre activité Créez et gérez votre compte DOCUMENTATION UTILISATEUR  Le Centre de Calcul de l’IN2P3 (ou CC-IN2P3) est une Unité d’Appui à la Recherche du CNRS (UAR6402) rattachée à l’IN2P3. Cet institut développe et coordonne les recherches ...


<a id="sec-3-4"></a>
### 3.4 Heading-aware chunking + contextual headers

This is the chunker we actually use to build the corpus. Two ideas combined:

1. **Never cross a heading boundary.** We walk each page's elements in order, tracking the
   current `h1 > h2 > h3` breadcrumb. Paragraphs/list-items/code blocks accumulate into a chunk
   until either the word budget is hit or a new heading starts — whichever comes first. A chunk
   therefore always belongs to exactly one section.
2. **Prepend a contextual header before embedding — but not before showing the chunk to a
   human or the LLM.** Each chunk gets an `embed_text` of the form
   `"{page title} — {heading breadcrumb}\n{chunk text}"`. This is what gets encoded by BGE-M3 in
   Section 4 (both the dense vector *and* the sparse lexical weights). The point: a short,
   otherwise-ambiguous chunk like *"The default is 3 and can be raised with `--cpus-per-task`"*
   becomes, once embedded, effectively *"SLURM Job Submission > Resource Requests — The default
   is 3 and can be raised with `--cpus-per-task`"* — which disambiguates it for retrieval without
   changing a single word of what's actually stored and shown as the source of truth
   (`chunk.text` stays untouched). This general technique — giving each chunk a bit of its
   surrounding context before indexing it — is sometimes called **contextual retrieval** in the
   RAG literature; what we're doing here is its cheapest version (a structural breadcrumb, free,
   no extra model call). Section 3.5 shows a stronger, LLM-generated version of the same idea as
   an optional bonus.

In [12]:
def heading_aware_chunk(elements: list, max_words: int = 180) -> List[dict]:
    '''
    Walk a page's ordered elements, grouping non-heading content under the current heading
    breadcrumb into chunks up to `max_words`. A chunk never spans two different headings.
    Returns a list of {"heading_path": [str, ...], "text": str} dicts.
    '''
    HEADING_LEVEL = {"h1": 1, "h2": 2, "h3": 3}
    chunks = []
    heading_stack: list[tuple[int, str]] = []  # [(level, text), ...]
    current_sentences: list[str] = []
    current_len = 0

    def flush():
        nonlocal current_sentences, current_len
        text = " ".join(current_sentences).strip()
        if text:
            chunks.append({"heading_path": [h[1] for h in heading_stack], "text": text})
        current_sentences, current_len = [], 0

    for el in elements:
        if el["tag"] in HEADING_LEVEL:
            flush()  # whatever we accumulated belongs to the PREVIOUS heading -- close it out
            level = HEADING_LEVEL[el["tag"]]
            heading_stack = [h for h in heading_stack if h[0] < level]  # pop deeper/equal headings
            heading_stack.append((level, el["text"]))
            continue

        sentences = [el["text"]] if el["tag"] in ("pre", "li") else re.split(r'(?<=[.!?])\s+', el["text"])
        for sent in sentences:
            sent_len = word_count(sent)
            if current_len + sent_len > max_words and current_sentences:
                flush()
            current_sentences.append(sent)
            current_len += sent_len

    flush()
    return chunks

# Try it on our sample document
sample_elements = raw_pages[0].get("elements") or [{"tag": "p", "text": raw_pages[0]["text"]}]
demo_chunks_v3 = heading_aware_chunk(sample_elements, max_words=180)
print(f"Heading-aware chunking -> {len(demo_chunks_v3)} chunks.")
for c in demo_chunks_v3[:2]:
    breadcrumb = " > ".join(c["heading_path"]) or "(no heading)"
    print(f"\n--- breadcrumb: {breadcrumb} ---\n{c['text'][:250]}...")

Heading-aware chunking -> 2 chunks.

--- breadcrumb: (no heading) ---
DOCUMENTATION UTILISATEUR Contactez le support Surveillez votre activité Créez et gérez votre compte...

--- breadcrumb: DOCUMENTATION UTILISATEUR  ---
Le Centre de Calcul de l’IN2P3 (ou CC-IN2P3) est une Unité d’Appui à la Recherche du CNRS (UAR6402) rattachée à l’IN2P3. Cet institut développe et coordonne les recherches en physique des particules, physique nucléaire et physique des astroparticules...


<a id="sec-3-5"></a>
### 3.5 Build the full chunk list

Now apply heading-aware chunking to every page, build the contextual `embed_text` for each
chunk, and keep a page-level text lookup for the "small-to-big" generation step in Section 7.

In [13]:
CHUNK_MAX_WORDS = 180

all_chunks: List[Chunk] = []
page_full_text_by_url = {}  # used in Section 7 for optional full-page context expansion

for doc_idx, page in enumerate(raw_pages):
    elements = page.get("elements") or [{"tag": "p", "text": page["text"]}]  # backward compatibility
    page_full_text_by_url[page["url"]] = page.get("text") or " ".join(e["text"] for e in elements)

    pieces = heading_aware_chunk(elements, CHUNK_MAX_WORDS)
    for i, piece in enumerate(pieces):
        heading_path_str = " > ".join(piece["heading_path"])
        header = f"{page['title']} — {heading_path_str}" if heading_path_str else page["title"]
        embed_text = f"{header}\n{piece['text']}"

        all_chunks.append(Chunk(
            chunk_id=f"doc{doc_idx}_chunk{i}",
            text=piece["text"],
            embed_text=embed_text,
            heading_path=heading_path_str,
            source_url=page["url"],
            source_title=page["title"],
            chunk_index=i,
        ))

print(f"Total chunks built: {len(all_chunks)} (from {len(raw_pages)} pages)")
print("Average words/chunk:", sum(word_count(c.text) for c in all_chunks) / len(all_chunks))
print("\nExample embed_text (what BGE-M3 will actually see):\n", all_chunks[0].embed_text[:300])

Total chunks built: 82 (from 15 pages)
Average words/chunk: 87.13414634146342

Example embed_text (what BGE-M3 will actually see):
 DOCUMENTATION UTILISATEUR — CC-IN2P3
DOCUMENTATION UTILISATEUR Contactez le support Surveillez votre activité Créez et gérez votre compte


In [14]:
# Save chunks + page-level text to disk so later sections (or a re-run) don't need to redo
# scraping + chunking.
import pickle

chunks_path = os.path.join(CORPUS_DIR, "chunks.pkl")
with open(chunks_path, "wb") as f:
    pickle.dump(all_chunks, f)

page_text_path = os.path.join(CORPUS_DIR, "page_full_text_by_url.pkl")
with open(page_text_path, "wb") as f:
    pickle.dump(page_full_text_by_url, f)

print("Saved chunks to:", chunks_path)
print("Saved page-level text lookup to:", page_text_path)

Saved chunks to: /home/anne/Documents/LaboBots/rag_workshop/corpus/chunks.pkl
Saved page-level text lookup to: /home/anne/Documents/LaboBots/rag_workshop/corpus/page_full_text_by_url.pkl


<a id="sec-3-6"></a>
### 3.6 (Optional, advanced) LLM-generated contextual summaries instead of a structural breadcrumb

If you previously had an LLM **summarize** each chunk (or each page) and embedded the summary
instead of the raw text — plus kept the original keywords around — you were doing a manual,
stronger version of the same idea as 3.4's breadcrumb. It's a legitimate and, in some cases,
more powerful technique. Worth being precise about when it helps and what it costs:

**When an LLM-generated summary/context genuinely helps, beyond the free breadcrumb trick:**
- Long, narratively written pages where the relevant fact is buried in a wall of prose the
  breadcrumb alone doesn't disambiguate.
- Very noisy source text (badly-OCR'd PDFs, auto-generated changelogs, dense tables) where a
  clean paraphrase embeds better than the raw noise.
- Aligning better with how people phrase questions: a query and a fluent, answer-shaped summary
  are often closer in embedding space than a query and a raw technical passage — this is the
  same intuition behind the "HyDE" (Hypothetical Document Embeddings) technique.

**What it costs, and the one rule that makes it safe:**
- One extra LLM call per chunk at indexing time (slow on CPU, and each call can occasionally
  drop or distort a specific number, command, or parameter — summaries are *lossy*).
- **The rule: a summary may be the thing you embed, but it must never be the thing you show the
  LLM at generation time.** Always keep `chunk.text` (the raw, original wording) as what gets
  put into the final prompt for generation — the summary is purely a *retrieval* aid, an index
  key, never the source of truth for the answer. Your original design (embed the summary, keep
  the original for generation, keep exact keywords too) already followed this rule correctly —
  that's the part that matters most, more than the summarization step itself.
- Your "keep the original keywords" instinct is, in effect, exactly what BGE-M3's sparse/lexical
  weights already give you automatically on the raw chunk (Section 4) — so if you adopt
  BGE-M3-style hybrid, you may not need a separate keyword-extraction step at all.

The cell below demonstrates the technique on a **handful of chunks only** (not the full corpus —
one Ollama call per chunk would be too slow for a live CPU-only session over hundreds of chunks;
treat this as something you'd run once, offline, ahead of time, the same way the pre-scraped
corpus was prepared).

In [15]:
# OPTIONAL / BONUS — demonstrates LLM-generated contextual summaries on a few chunks.
# Not used in the main pipeline below; Section 4 embeds the free breadcrumb-based `embed_text`.
import ollama

CONTEXT_MODEL = "llama3.2:3b"

def llm_contextualize(chunk_text: str, page_title: str, model: str = CONTEXT_MODEL) -> str:
    '''
    Ask a small local LLM for a one-sentence statement of what this chunk is about, to prepend
    before embedding -- a stronger (but slower, and slightly lossy-risk) alternative to the
    structural breadcrumb in 3.4.
    '''
    prompt = (
        f"Document: {page_title}\n\nPassage:\n{chunk_text}\n\n"
        "In ONE short sentence, state what this passage is about, so it can be found by someone "
        "searching for this topic. Do not answer questions, just describe the topic."
    )
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"].strip()

for demo_chunk in all_chunks[:3]:
    context_sentence = llm_contextualize(demo_chunk.text, demo_chunk.source_title)
    print(f"Chunk: {demo_chunk.text[:100]}...")
    print(f"LLM-generated context: {context_sentence}")
    print(f"-> would embed: '{context_sentence}\\n{demo_chunk.text[:80]}...'\\n")

Chunk: DOCUMENTATION UTILISATEUR Contactez le support Surveillez votre activité Créez et gérez votre compte...
LLM-generated context: User documentation for the CC-IN2P3 platform.
-> would embed: 'User documentation for the CC-IN2P3 platform.\nDOCUMENTATION UTILISATEUR Contactez le support Surveillez votre activité Créez e...'\n
Chunk: Le Centre de Calcul de l’IN2P3 (ou CC-IN2P3) est une Unité d’Appui à la Recherche du CNRS (UAR6402) ...
LLM-generated context: This passage is a user documentation guide for the Centre de Calcul de l'IN2P3 (CC-IN2P3) research facility.
-> would embed: 'This passage is a user documentation guide for the Centre de Calcul de l'IN2P3 (CC-IN2P3) research facility.\nLe Centre de Calcul de l’IN2P3 (ou CC-IN2P3) est une Unité d’Appui à la Recherch...'\n
Chunk: Compte utilisateur...
LLM-generated context: This passage is about a user account.
-> would embed: 'This passage is about a user account.\nCompte utilisateur...'\n


<a id="sec-4"></a>
## 4. Embeddings — dense *and* sparse, from a single model (BGE-M3)

<a id="sec-4-1"></a>
### 4.1 Why BGE-M3?

For a technical documentation corpus in a mixed French/English lab environment, we want an
embedding model that is:

- **Multilingual** (your lab's docs are rarely 100% in one language),
- **Good at both short queries and longer passages**,
- Able to produce **more than one type of representation**, because that's exactly what our
  hybrid retriever needs.

[BGE-M3](https://huggingface.co/BAAI/bge-m3) (BAAI General Embedding, "M3" = Multi-Lingual,
Multi-Functionality, Multi-Granularity) is a great fit: from a single forward pass over a piece
of text, it produces **three** representations:

| Representation | What it captures | How we'll use it |
|---|---|---|
| **Dense vector** (1024-dim) | Overall semantic meaning | Semantic ("meaning") search |
| **Sparse / lexical weights** | Importance weight per token (like a smarter TF-IDF/BM25) | Exact keyword / acronym matching |
| **ColBERT-style multi-vector** | Fine-grained token-level interaction | *Not used in this workshop* (more advanced, heavier to compute) |

This is exactly the "dense + sparse" hybrid you want, from **one model**, which keeps the
pipeline simple: one encoding pass, two scores to combine later.

> **Why not just use dense embeddings?** Dense embeddings are excellent at "meaning" but often
> miss exact technical terms — job names, error codes, environment variable names, cluster
> partition names. A user searching for `SLURM_ARRAY_TASK_ID` wants an *exact* match, and sparse
> retrieval is much better at that than pure semantic similarity. Combining both gives you the
> best of both worlds — this is the whole point of hybrid search.

**Where to check a model's actual embedding size yourself** (useful beyond this workshop, for any
model you're evaluating): the [BGE-M3 model card on Hugging Face](https://huggingface.co/BAAI/bge-m3)
states it directly ("1024-dim dense vectors"). For a model served through Ollama instead, run
`ollama show <model>` (or `GET /api/show`) — it prints `embedding_length` alongside parameter
count and quantization. Dense size is always fixed per model (BGE-M3: 1024, every chunk, always);
sparse "size" isn't a fixed number at all — it's a dict over your tokenizer's vocabulary, one
weight per token that actually appears, which is why `lexical_weights` below is a list of *dicts*,
not a list of same-length vectors like `dense_vecs` is.

**Architecture, visually**: BGE-M3's own paper — [*BGE M3-Embedding: Multi-Lingual,
Multi-Functionality, Multi-Granularity Text Embeddings*](https://arxiv.org/abs/2402.03216) —
has a figure showing exactly how one shared Transformer backbone (1.0's primer) produces all
three representations (dense / sparse / ColBERT) from a single forward pass; the model card link
above summarizes the same idea without needing to read the full paper.

<a id="sec-4-2"></a>
### 4.2 Model size / performance note for CPU-only laptops

BGE-M3 has ~568M parameters (~2.2 GB download). On CPU, encoding ~1,000 chunks of ~180 words
takes roughly 5–10 minutes on a modern laptop (varies a lot by CPU). For the live session with a
larger corpus, the instructor may hand out **pre-computed embeddings** (`embeddings.npz` +
`lexical_weights.pkl`) so you can skip straight to Section 5 if your machine is slow. The code
below is written so it transparently uses a local cache if present.

If your laptop really struggles, swap `BGE_MODEL_NAME` below for a lighter alternative such as
`intfloat/multilingual-e5-small` (dense-only — you would then need a separate BM25 sparse index,
shown as a fallback in 4.4).

> **No Hugging Face account needed.** `BAAI/bge-m3` is a public, non-gated model — the line below
> downloads its weights anonymously over plain HTTPS the first time it runs, then caches them
> locally (by default under `~/.cache/huggingface/`). You do **not** need to sign up, log in, or
> set a token for anything in this notebook. (Some *other* models on Hugging Face are "gated"
> and do require a free account + accepting a license — just not this one.)
>
> **Could I instead serve embeddings from Ollama, like the LLM?** Ollama does host `bge-m3`
> (`ollama pull bge-m3`) and it's just as account-free as the LLM — but its `/api/embeddings`
> endpoint only returns the **dense** vector. The sparse/lexical output that makes our hybrid
> search work comes from the `FlagEmbedding` library specifically calling BGE-M3's dedicated
> sparse head — Ollama's API doesn't expose that (confirmed: there's an open, unresolved request
> for it on Ollama's own GitHub). So switching embeddings to Ollama isn't a drop-in swap; it
> would mean giving up native sparse and falling back to classic BM25 for the sparse side
> (already coded as an alternative in 4.4) instead. Not needed here since BGE-M3 via
> `FlagEmbedding` already requires no account — but worth knowing if you ever want a
> Ollama-only stack with zero non-Ollama dependencies for the embedding side too.

In [16]:
from FlagEmbedding import BGEM3FlagModel

BGE_MODEL_NAME = "BAAI/bge-m3"

print("Loading BGE-M3 (first run downloads ~2.2 GB, then it's cached locally)...")
bge_model = BGEM3FlagModel(BGE_MODEL_NAME, use_fp16=False)  # fp16 mainly helps on GPU
print("Model loaded.")

Loading BGE-M3 (first run downloads ~2.2 GB, then it's cached locally)...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded.


In [17]:
# --- Encode a couple of sample texts to see the shape of what BGE-M3 gives us ---
# NOTE: we encode `embed_text` (contextual header + chunk), not the raw `text` -- see Section 3.4.
sample_texts = [all_chunks[0].embed_text, all_chunks[1].embed_text if len(all_chunks) > 1 else all_chunks[0].embed_text]

sample_output = bge_model.encode(
    sample_texts,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,   # we don't need the heaviest representation for this workshop
)

print("Dense vector shape:", sample_output["dense_vecs"].shape)
print("\nSparse weights for chunk 0 (top 10 highest-weight tokens):")
lex_weights = sample_output["lexical_weights"][0]
top_tokens = sorted(lex_weights.items(), key=lambda kv: kv[1], reverse=True)[:10]
for token_id, weight in top_tokens:
    token_str = bge_model.tokenizer.decode([int(token_id)])
    print(f"  token='{token_str}'  weight={weight:.4f}")

Dense vector shape: (2, 1024)

Sparse weights for chunk 0 (top 10 highest-weight tokens):
  token='compte'  weight=0.2627
  token='support'  weight=0.2393
  token='EUR'  weight=0.1953
  token='SAT'  weight=0.1837
  token='MENT'  weight=0.1754
  token='activité'  weight=0.1706
  token='CC'  weight=0.1704
  token='Cré'  weight=0.1702
  token='ILI'  weight=0.1681
  token='U'  weight=0.1641


<a id="sec-4-3"></a>
### 4.3 Encode the full corpus

We now encode **all** chunks in batches (batching keeps memory usage bounded and speeds things
up). We encode `chunk.embed_text` (contextual header + raw text, from Section 3.4) — **not**
`chunk.text` alone — so both the dense vector and the sparse lexical weights benefit from the
extra section context. We keep:

- `dense_embeddings`: a NumPy array of shape `(n_chunks, 1024)`
- `sparse_weights`: a Python list of dicts (one dict per chunk: `token_id -> weight`)

Both are saved to disk, so you never have to re-run this expensive cell unless the corpus changes.

> ⚠️ **The one gotcha with any on-disk cache**: if you re-scrape, re-chunk, or otherwise change
> `all_chunks` (a different `MAX_PAGES`, a different corpus file, restarting mid-workshop with
> the instructor's full corpus instead of your live sample...) **and then re-run this cell**, a
> stale `embeddings.npz` from the *previous* corpus will get loaded instead of being recomputed —
> because the cell only checks whether the cache file *exists*, not whether it still matches the
> current `all_chunks`. That mismatch is exactly what produces a
> `ValueError: Unequal lengths for fields: ids: 82, ..., embeddings: 51` a few cells later in
> Section 5 (`collection.add(...)`) — it means yesterday's 51-chunk cache is being fed into
> today's 82-chunk corpus. **It has nothing to do with the Chroma collection itself** (deleting
> and recreating the collection, as Section 5.2 already does, does not fix this — the bug is
> upstream, in this cache). The cell below now checks the cached array's length against
> `len(all_chunks)` and automatically recomputes if they don't match, so this should no longer
> bite you silently — but if you ever see that exact error again, the fix is simply to delete
> `rag_workshop/corpus/embeddings.npz` and `lexical_weights.pkl` and re-run this cell.

In [18]:
import numpy as np
from tqdm.auto import tqdm

EMB_CACHE_PATH = os.path.join(CORPUS_DIR, "embeddings.npz")
LEX_CACHE_PATH = os.path.join(CORPUS_DIR, "lexical_weights.pkl")

BATCH_SIZE = 8  # keep this small on CPU-only laptops; raise it if you have more RAM/cores

def _load_cache_if_valid(expected_n: int):
    '''
    Load cached embeddings only if they exist, aren't corrupted, AND were computed for the SAME
    corpus we currently have in memory (same number of chunks). Returns (dense_embeddings,
    sparse_weights) or None. The size check is what prevents a stale cache from a previous,
    different-sized corpus from silently getting fed into Chroma later (which fails with a
    confusing "Unequal lengths" error); the try/except is what prevents a truncated/corrupted
    cache file (e.g. from an interrupted previous run) from crashing this cell instead of just
    recomputing.
    '''
    if not (os.path.exists(EMB_CACHE_PATH) and os.path.exists(LEX_CACHE_PATH)):
        return None
    try:
        cached_dense = np.load(EMB_CACHE_PATH)["dense"]
        with open(LEX_CACHE_PATH, "rb") as f:
            cached_sparse = pickle.load(f)
    except Exception as e:
        print(f"Found a cache, but it's corrupted/unreadable ({e}) -- recomputing instead.")
        return None
    if cached_dense.shape[0] != expected_n or len(cached_sparse) != expected_n:
        print(
            f"Found a cache, but it has {cached_dense.shape[0]} embeddings for a corpus that "
            f"now has {expected_n} chunks -- it's stale (from an earlier scrape/chunking run). "
            "Recomputing instead of using it."
        )
        return None
    return cached_dense, cached_sparse

cached = _load_cache_if_valid(len(all_chunks))

if cached is not None:
    print("Found a valid, matching cache -- loading instead of recomputing.")
    dense_embeddings, sparse_weights = cached
else:
    all_texts = [c.embed_text for c in all_chunks]
    dense_list, sparse_list = [], []

    for i in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="Encoding chunks"):
        batch = all_texts[i:i + BATCH_SIZE]
        out = bge_model.encode(batch, return_dense=True, return_sparse=True, return_colbert_vecs=False)
        dense_list.append(out["dense_vecs"])
        sparse_list.extend(out["lexical_weights"])

    dense_embeddings = np.vstack(dense_list)
    sparse_weights = sparse_list

    np.savez_compressed(EMB_CACHE_PATH, dense=dense_embeddings)
    with open(LEX_CACHE_PATH, "wb") as f:
        pickle.dump(sparse_weights, f)

assert dense_embeddings.shape[0] == len(all_chunks), (
    f"dense_embeddings has {dense_embeddings.shape[0]} rows but all_chunks has {len(all_chunks)} "
    "-- something is still out of sync; re-run this cell after checking Section 3.5 ran with the "
    "corpus you intended."
)
assert len(sparse_weights) == len(all_chunks), "sparse_weights length does not match all_chunks."

print("Dense embeddings shape:", dense_embeddings.shape)
print("Sparse weights: one dict per chunk, e.g. chunk 0 has", len(sparse_weights[0]), "weighted tokens.")

Found a valid, matching cache -- loading instead of recomputing.
Dense embeddings shape: (82, 1024)
Sparse weights: one dict per chunk, e.g. chunk 0 has 29 weighted tokens.


<a id="sec-4-4"></a>
### 4.4 Fallback: classic BM25 sparse index (if you're not using BGE-M3's native sparse output)

If you swapped in a dense-only embedding model, here is how you'd build a classic sparse index
with `rank_bm25` instead. **You can skip this cell if you're using BGE-M3's lexical weights above** —
it's shown for completeness, since BM25 is the "classic" sparse method you'll see referenced
everywhere in RAG literature.

In [19]:
# OPTIONAL fallback — not used in the main path if BGE-M3 sparse weights are available.
# Uncomment to try it.

# from rank_bm25 import BM25Okapi
#
# tokenized_corpus = [c.text.lower().split() for c in all_chunks]
# bm25_index = BM25Okapi(tokenized_corpus)
#
# def bm25_search(query: str, top_k: int = 5):
#     tokenized_query = query.lower().split()
#     scores = bm25_index.get_scores(tokenized_query)
#     top_indices = np.argsort(scores)[::-1][:top_k]
#     return [(all_chunks[i], scores[i]) for i in top_indices]
#
# print("BM25 fallback ready (uncomment the code above to use it).")
print("Skipped — using BGE-M3 native sparse weights as our main sparse representation.")

Skipped — using BGE-M3 native sparse weights as our main sparse representation.


<a id="sec-5"></a>
## 5. Building the hybrid vector store

<a id="sec-5-1"></a>
### 5.1 Why ChromaDB?

[ChromaDB](https://www.trychroma.com/) is an **embedded** vector database — it runs in-process,
no server to start, no Docker required, and persists to a local folder. This makes it ideal for
a from-scratch pedagogical setting: you can inspect exactly what goes in and comes out.

> **Note on hybrid search and ChromaDB**: as of the version used in this workshop, Chroma stores
> and indexes **dense vectors natively**, but it does **not** natively store/query sparse
> (lexical) vectors the way a dedicated hybrid engine like Qdrant does. This is a deliberate
> pedagogical choice: we will store the dense vectors *in* Chroma, and keep the sparse
> (lexical) weights *alongside* in a simple Python structure, then **fuse the two ourselves**
> in Section 6. This forces us to understand exactly what "hybrid" means instead of trusting a
> black-box `hybrid_search()` call.
>
> If you want a vector store with **native** hybrid support (dense + sparse fused server-side),
> see the optional **Qdrant** bonus module later in the course — it requires Docker and a bit
> more setup, which is why it's presented as a follow-up rather than the main path.

<a id="sec-5-2"></a>
### 5.2 Create a persistent Chroma collection

<p align="center">
  <img src="https://github.com/chroma-core/chroma/raw/main/docs/assets/chroma-wordmark-color.png" alt="ChromaDB logo" width="260"><br>
  <sub><i>ChromaDB — la base vectorielle embarquée utilisée dans ce notebook</i></sub>
</p>

<p align="center">
  <img src="https://media.geeksforgeeks.org/wp-content/uploads/20250210184749053767/What-is-RAG_.webp" alt="What is RAG diagram" width="460"><br>
  <sub><i>Retrieve · Augment · Generate — Chroma joue le rôle de « Vector DB » dans ce schéma — [GeeksforGeeks](https://www.geeksforgeeks.org/nlp/what-is-retrieval-augmented-generation-rag/)</i></sub>
</p>


In [20]:
import chromadb

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

COLLECTION_NAME = "ccin2p3_docs"

# Recreate the collection cleanly each time we run this notebook end-to-end
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # explicitly use cosine similarity, matches BGE-M3 training
)
print(f"Created Chroma collection '{COLLECTION_NAME}' (cosine similarity).")

Created Chroma collection 'ccin2p3_docs' (cosine similarity).


<a id="sec-5-3"></a>
### 5.3 Populate the collection

We insert, for every chunk: its **id**, its **dense embedding** (precomputed in Section 4 —
we pass our own vectors rather than letting Chroma compute them, since we already have BGE-M3
embeddings), the **raw text**, and **metadata** (source URL, title, chunk index).

We also keep a separate Python dictionary `sparse_index` mapping `chunk_id -> lexical_weights`,
since that part lives outside Chroma.

In [21]:
CHROMA_BATCH_SIZE = 200  # Chroma handles large batches fine; this just avoids one giant call

ids = [c.chunk_id for c in all_chunks]
documents = [c.text for c in all_chunks]
metadatas = [
    {
        "source_url": c.source_url,
        "source_title": c.source_title,
        "chunk_index": c.chunk_index,
        "heading_path": c.heading_path,
    }
    for c in all_chunks
]
embeddings_list = dense_embeddings.tolist()

# Defense in depth: fail loudly and clearly here rather than with Chroma's internal
# "Unequal lengths" error, in case all_chunks / dense_embeddings ever get out of sync again
# (e.g. Section 3 was re-run with a different corpus after Section 4 already cached embeddings).
assert len(ids) == len(embeddings_list) == len(documents) == len(metadatas), (
    f"Mismatched lengths before inserting into Chroma: ids={len(ids)}, "
    f"embeddings={len(embeddings_list)}, documents={len(documents)}, metadatas={len(metadatas)}. "
    "This usually means Section 4's embeddings.npz cache is stale relative to the current "
    "all_chunks -- re-run Section 4.3 (it will detect and recompute automatically now)."
)

for i in tqdm(range(0, len(ids), CHROMA_BATCH_SIZE), desc="Inserting into Chroma"):
    collection.add(
        ids=ids[i:i + CHROMA_BATCH_SIZE],
        embeddings=embeddings_list[i:i + CHROMA_BATCH_SIZE],
        documents=documents[i:i + CHROMA_BATCH_SIZE],
        metadatas=metadatas[i:i + CHROMA_BATCH_SIZE],
    )

print(f"Inserted {collection.count()} chunks into Chroma.")

# Sparse index lives outside Chroma: a plain dict keyed by chunk_id
sparse_index = {chunk_id: weights for chunk_id, weights in zip(ids, sparse_weights)}
print(f"Sparse index built for {len(sparse_index)} chunks (kept in memory / can be pickled).")

Inserting into Chroma:   0%|          | 0/1 [00:00<?, ?it/s]

Inserted 82 chunks into Chroma.
Sparse index built for 82 chunks (kept in memory / can be pickled).


In [22]:
# Sanity check: a pure dense query, without any fusion yet, just to confirm the store works
test_query = "How do I submit a job with SLURM?"
test_query_emb = bge_model.encode([test_query], return_dense=True, return_sparse=False)["dense_vecs"][0]

result = collection.query(query_embeddings=[test_query_emb.tolist()], n_results=3)
for rank, (doc, meta, dist) in enumerate(zip(result["documents"][0], result["metadatas"][0], result["distances"][0]), 1):
    print(f"#{rank}  distance={dist:.4f}  source={meta['source_title']}")
    print("   ", doc[:150].replace(chr(10), ' '), "...\n")

#1  distance=0.3947  source=La plateforme de calcul — CC-IN2P3
    Son rôle est de recevoir les jobs soumis par les utilisateurs, de les ordonnancer et de les soumettre pour exécution sur un serveur de calcul appropri ...

#2  distance=0.4945  source=Compte utilisateur — CC-IN2P3
    Lors de l’étape 4. , il vous est demandé de sélectionner votre structure de recherche (SR) depuis un menu déroulant. Si votre SR est associée à une co ...

#3  distance=0.5467  source=Compte utilisateur — CC-IN2P3
    Si, dans le cas contraire, vous n’avez pas trouvé votre SR dans le menu ou elle n’est pas associée à une collaboration répertoriée, vous aurez créé un ...



<a id="sec-6"></a>
## 6. Hybrid retrieval — combining dense and sparse search with Reciprocal Rank Fusion

<a id="sec-6-1"></a>
### 6.1 The problem with combining raw scores

Dense search gives you **cosine similarities** (roughly 0 to 1). Sparse/lexical search gives you
**dot-product scores** on a completely different scale (can be any positive number, often much
larger than 1). You **cannot** just add these two scores together — one would dominate the other
for no meaningful reason.

<a id="sec-6-2"></a>
### 6.2 Reciprocal Rank Fusion (RRF)

RRF sidesteps the scale problem entirely by ignoring the raw scores and using only **ranks**.
For each retrieval method, look at the rank position of each candidate (1st, 2nd, 3rd...), then
combine with the formula:

$$\text{RRF}(d) = \sum_{\text{method} \in \{\text{dense}, \text{sparse}\}} \frac{1}{k + \text{rank}_{\text{method}}(d)}$$

where `k` is a small constant (commonly `k = 60`) that dampens the influence of very low ranks
and prevents division-by-zero issues. A document that ranks well in **both** lists gets a high
combined score; a document that only appears in one list still gets *some* credit, proportional
to how high it ranked there.

This is simple, scale-free, and it's what many production hybrid search systems actually use
under the hood (including, e.g., Elasticsearch's hybrid retriever).

<a id="sec-6-3"></a>
### 6.3 Implementing sparse (lexical) scoring manually

Since our sparse representation lives outside Chroma, we compute lexical matching scores
ourselves using BGE-M3's own scoring utility (a weighted dot-product over shared tokens between
query and chunk).

> ⚠️ **A subtle correctness bug to avoid**: RRF (6.2) fuses lists by *rank position only* — it
> never looks at the underlying score. If you naively return the top `k` sparse results without
> checking their score, a query with very little lexical overlap in the corpus will still return
> `k` chunks (padded with genuinely irrelevant, zero-score ones just because *something* has to
> fill the slots), and RRF will hand those zero-score chunks a real ranking bonus for a match
> that doesn't exist. `sparse_search` below explicitly drops zero-score chunks *before*
> truncating to `top_k`, so a chunk only ever influences the fused ranking if it actually shares
> some lexical content with the query.

<p align="center">
  <img src="https://media.geeksforgeeks.org/wp-content/uploads/20200911171455/UntitledDiagram2.png" alt="Cosine similarity between two vectors" width="260"><br>
  <sub><i>Similarité cosinus entre deux vecteurs — la mesure utilisée par la recherche dense — [GeeksforGeeks](https://www.geeksforgeeks.org/dbms/cosine-similarity/)</i></sub>
</p>


In [23]:
def sparse_search(query_lexical_weights: dict, sparse_index: dict, top_k: int = 10):
    '''
    Score every chunk in `sparse_index` against the query's lexical weights using
    BGE-M3's own lexical matching function (weighted overlap of token ids),
    then return the top_k (chunk_id, score) pairs, best first.

    IMPORTANT: we drop zero-score chunks before truncating to top_k. RRF (below) only looks at
    *rank position*, not the underlying score -- so if fewer than top_k chunks have any real
    lexical overlap with the query, padding the list with zero-score chunks just to fill top_k
    would hand them a real RRF bonus for a match that doesn't exist. A chunk with score 0.0 has
    no business influencing the fused ranking at all.
    '''
    scores = [
        (chunk_id, bge_model.compute_lexical_matching_score(query_lexical_weights, chunk_weights))
        for chunk_id, chunk_weights in sparse_index.items()
    ]
    scores = [(cid, s) for cid, s in scores if s > 0.0]
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


def dense_search(query_dense_vec, collection, top_k: int = 10):
    '''
    Query Chroma for the top_k nearest chunks by cosine similarity.
    Returns a list of (chunk_id, similarity) pairs, best first.
    Chroma returns *distances*; for cosine space, similarity = 1 - distance.
    '''
    result = collection.query(query_embeddings=[query_dense_vec.tolist()], n_results=top_k)
    chunk_ids = result["ids"][0]
    distances = result["distances"][0]
    similarities = [1 - d for d in distances]
    return list(zip(chunk_ids, similarities))


def reciprocal_rank_fusion(ranked_lists: list, k: int = 60):
    '''
    ranked_lists: a list of ranked lists, each a list of (doc_id, score) pairs (best first).
    Returns a single fused ranking as a list of (doc_id, rrf_score) pairs, best first.
    '''
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for rank, (doc_id, _score) in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused

print("Hybrid search functions defined: dense_search, sparse_search, reciprocal_rank_fusion.")

Hybrid search functions defined: dense_search, sparse_search, reciprocal_rank_fusion.


In [24]:
def hybrid_retrieve(query: str, top_k_each: int = 10, top_k_final: int = 5):
    '''
    Full hybrid retrieval pipeline:
      1. Encode the query once with BGE-M3 (dense + sparse in one pass).
      2. Run dense search against Chroma.
      3. Run sparse search against our in-memory lexical index.
      4. Fuse both ranked lists with RRF.
      5. Return the top_k_final fused results, enriched with their original text/metadata.
    '''
    query_encoded = bge_model.encode([query], return_dense=True, return_sparse=True)
    query_dense = query_encoded["dense_vecs"][0]
    query_sparse = query_encoded["lexical_weights"][0]

    dense_results = dense_search(query_dense, collection, top_k=top_k_each)
    sparse_results = sparse_search(query_sparse, sparse_index, top_k=top_k_each)

    fused = reciprocal_rank_fusion([dense_results, sparse_results], k=60)[:top_k_final]

    # Also compute the raw cosine similarity of each fused result to the query,
    # which we'll use in Section 7 to decide how confident we are in the retrieval.
    dense_sim_lookup = dict(dense_results)

    enriched = []
    chunk_lookup = {c.chunk_id: c for c in all_chunks}
    for chunk_id, rrf_score in fused:
        chunk = chunk_lookup[chunk_id]
        cosine_sim = dense_sim_lookup.get(chunk_id, None)  # may be None if only found via sparse
        enriched.append({
            "chunk_id": chunk_id,
            "text": chunk.text,
            "source_url": chunk.source_url,
            "source_title": chunk.source_title,
            "heading_path": chunk.heading_path,
            "rrf_score": rrf_score,
            "cosine_similarity": cosine_sim,
        })
    return enriched


# Try it out
demo_results = hybrid_retrieve("How do I submit a job with SLURM?", top_k_each=10, top_k_final=5)
for rank, r in enumerate(demo_results, 1):
    cos = f"{r['cosine_similarity']:.4f}" if r["cosine_similarity"] is not None else "n/a (sparse-only)"
    label = f"{r['source_title']} — {r['heading_path']}" if r["heading_path"] else r["source_title"]
    print(f"#{rank}  RRF={r['rrf_score']:.4f}  cosine={cos}  source={label}")
    print("   ", r["text"][:150].replace(chr(10), " "), "...\n")

#1  RRF=0.0328  cosine=0.6053  source=La plateforme de calcul — CC-IN2P3 — La plateforme de calcul 
    Son rôle est de recevoir les jobs soumis par les utilisateurs, de les ordonnancer et de les soumettre pour exécution sur un serveur de calcul appropri ...

#2  RRF=0.0315  cosine=0.4519  source=Espaces de stockage — CC-IN2P3 — Espaces de stockage  > Espaces de travail 
    En plus de votre HOME et des répertoires THRONG et GROUP, vous pouvez utiliser depuis les serveurs d’accueil un répertoire de travail /scratch pour y  ...

#3  RRF=0.0313  cosine=0.4496  source=La plateforme de calcul — CC-IN2P3 — La plateforme de calcul 
    Plateforme Jupyter Notebooks Gestion des noyaux Jupyter Utilisation de Dask Exécution Conseils pratiques Gestion des noyaux Jupyter Utilisation de Das ...

#4  RRF=0.0161  cosine=0.5055  source=Compte utilisateur — CC-IN2P3 — Compte utilisateur  > Demander un compte  > Selection de la structure de recherche 
    Lors de l’étape 4. , il vous est demandé 

<a id="sec-6-4"></a>
### 6.4 Compare: dense-only vs sparse-only vs hybrid

A good pedagogical exercise: try a query containing a very specific technical term (an exact
command, an environment variable, an error message) and compare the three retrieval modes.
You should observe sparse (lexical) search doing noticeably better on exact terms, and dense
search doing better on paraphrased / conceptual questions. Hybrid should be robust to both.

In [25]:
def compare_retrieval_modes(query: str, top_k: int = 3):
    query_encoded = bge_model.encode([query], return_dense=True, return_sparse=True)
    query_dense = query_encoded["dense_vecs"][0]
    query_sparse = query_encoded["lexical_weights"][0]

    dense_only = dense_search(query_dense, collection, top_k=top_k)
    sparse_only = sparse_search(query_sparse, sparse_index, top_k=top_k)
    hybrid = hybrid_retrieve(query, top_k_each=10, top_k_final=top_k)

    chunk_lookup = {c.chunk_id: c for c in all_chunks}

    print(f"Query: {query!r}\n")
    print("--- DENSE ONLY ---")
    for cid, score in dense_only:
        print(f"  {score:.4f}  {chunk_lookup[cid].source_title}  |  {chunk_lookup[cid].text[:80]}...")
    print("\n--- SPARSE ONLY ---")
    for cid, score in sparse_only:
        print(f"  {score:.4f}  {chunk_lookup[cid].source_title}  |  {chunk_lookup[cid].text[:80]}...")
    print("\n--- HYBRID (RRF) ---")
    for r in hybrid:
        print(f"  {r['rrf_score']:.4f}  {r['source_title']}  |  {r['text'][:80]}...")

# Try your own queries here — swap in something specific to what you scraped!
compare_retrieval_modes("SLURM job array environment variable")

Query: 'SLURM job array environment variable'

--- DENSE ONLY ---
  0.5309  La plateforme de calcul — CC-IN2P3  |  Son rôle est de recevoir les jobs soumis par les utilisateurs, de les ordonnance...
  0.5065  Environnement logiciel — CC-IN2P3  |  Dans le cas d’une installation personnalisée, la tentative d’ajout d’un module p...
  0.4813  Espaces de stockage — CC-IN2P3  |  En plus de votre HOME et des répertoires THRONG et GROUP, vous pouvez utiliser d...

--- SPARSE ONLY ---
  0.1440  La plateforme de calcul — CC-IN2P3  |  Son rôle est de recevoir les jobs soumis par les utilisateurs, de les ordonnance...
  0.0806  Espaces de stockage — CC-IN2P3  |  Le répertoire HOME est dédié au stockage de données personnelles, et est sauvega...
  0.0532  Environnement logiciel — CC-IN2P3  |  Dans le cas d’une installation personnalisée, la tentative d’ajout d’un module p...

--- HYBRID (RRF) ---
  0.0328  La plateforme de calcul — CC-IN2P3  |  Son rôle est de recevoir les jobs soumis par les utili

<a id="sec-7"></a>
## 7. Conversational scenario driven by similarity, and generation with a local LLM

<a id="sec-7-1"></a>
### 7.1 Why gate the answer on the similarity score?

A RAG system that *always* answers, even when nothing relevant was retrieved, will confidently
hallucinate. We use the **top cosine similarity score** from Section 6 as a simple, transparent
confidence signal to decide between four conversational behaviors, each also using a **different
LLM sampling configuration** — the intuition: the more we trust the retrieved evidence, the less
"creative freedom" we want the model to have when turning it into prose.

| Top cosine similarity | Mode | `temperature` | `top_p` | `top_k` |
|---|---|---|---|---|
| ≥ 0.80 | RAG only — near-extractive, minimal paraphrase drift | 0.1 | 0.7 | 10 |
| 0.60 – 0.80 | RAG + light synthesis | 0.3 | 0.7 | 10 |
| 0.50 – 0.60 | RAG, but hedge explicitly | 0.7 | 0.7 | 10 |
| < 0.50 | No usable evidence — LLM answers from general knowledge, clearly flagged as such | 0.7 | 0.9 | 40 |

This lines up with a natural intuition: high confidence → keep the model close to a near-greedy
decoding of the evidence (`temperature=0.1`); low confidence → either hedge more visibly, or (new
compared to a simple refuse/answer switch) let the LLM fall back to its own general knowledge —
**as long as that fallback is clearly labeled as not coming from your lab's documentation**. Widen
`top_p`/`top_k` a bit in that last band too, since the model isn't constrained by retrieved text
anymore and default OpenAI/Ollama-style values (`top_p≈0.9`, `top_k≈40`) are more appropriate than
the narrow, evidence-following settings used in the other three bands.

> ⚠️ **Two things this does *not* buy you — worth being precise about, because it's an easy
> trap:**
> 1. **Cosine similarity is not confidence that the passage actually answers the question.** It
>    only measures topical proximity between the query and a chunk. A chunk can score 0.85
>    because it's clearly about the same *topic* as the question, while still not containing the
>    specific fact asked for. The banding above is a reasonable, cheap proxy — not a guarantee.
>    A stronger (and more expensive) alternative is an explicit **answerability check**: a short
>    extra LLM call, or a lightweight NLI-style classifier, asking "does this passage actually
>    answer this question?" before deciding the band. Worth doing if you find the cosine proxy
>    misfires often on your corpus; out of scope for the main path today.
> 2. **Lowering `temperature` does not make the model more truthful, only more deterministic.**
>    Temperature narrows the sampling distribution over next tokens; it doesn't make the model
>    consult the source text more carefully. A low-temperature generation can still confidently
>    state something the context doesn't support — asking the model to "cite a source" does not
>    guarantee the citation actually supports the sentence it's attached to. The main defenses
>    against that stay what they were before this section: a system prompt that constrains the
>    model to the provided context, good retrieval (so the context is actually relevant), and a
>    human reviewing anything that matters. Sampling parameters are a UX polish layer on top of
>    that — matching the *style* of the answer to how much we trust the evidence — not a
>    substitute for it.

<a id="sec-7-2"></a>
### 7.2 The role of the LLM here

Note the framing from the course objectives: in the top three bands, the LLM's job is **not** to
invent facts — the facts come entirely from the retrieved chunks; its job is to turn them into a
**fluent, well-organized, cited answer**. In the bottom band, we're explicitly and visibly
switching modes: the LLM is now answering from its own training data, not your documentation, and
the answer must say so.

<a id="sec-7-3"></a>
### 7.3 "Small-to-big": search with chunks, generate with more context

This is where we close the loop on the chunking discussion from Section 3.1. We search with
small, precise chunks (good for hybrid RRF scoring), but before handing context to the LLM, we
can **expand** each retrieved chunk back out — up to its full page — using the
`page_full_text_by_url` lookup saved in Section 3.5. This gives the generator the surrounding
context a lone chunk lacks, without ever having searched over diluted whole-page embeddings.

`EXPAND_TOP_N_TO_FULL_PAGE` controls how many of the top results get expanded (the rest stay as
plain chunks, to keep the prompt from ballooning): expanding *every* result to a full page
defeats the purpose (you're back to feeding the LLM everything, most of it irrelevant, and you'll
hit the small local model's context window fast) — expanding just the top 1-2 highest-confidence
hits is usually the sweet spot. Set it to `0` to disable expansion entirely and see the
difference for yourself.

> ⚠️ **A real failure mode of naive expansion**: if we simply keep the *first* N characters of
> the expanded page, and the actual matched passage happens to sit further down, we silently
> **throw away the very evidence retrieval just found** and hand the LLM a truncated page that
> may not even contain it anymore. The fix below keeps the retrieved chunk's own text **always**
> present in the context block, and *adds* nearby page content around it (rather than replacing
> it outright) — so expansion can only add context, never lose the evidence that got the chunk
> retrieved in the first place.

In [ ]:
import ollama

GENERATION_MODEL = "llama3.2:3b"

# --- Confidence bands: starting points, to be calibrated on your own corpus (see 7.5) ---
def sampling_for_similarity(top_similarity: float) -> tuple[str, dict]:
    '''
    Map a top cosine similarity to (mode_name, sampling_options). See 7.1's table.
    Returned dict keys match Ollama's `options` (and are forwarded as-is to LiteLLM -> Ollama
    in notebook 2 / the Thunderbird plugin, since LiteLLM passes non-OpenAI-standard params
    like `top_k` straight through to the underlying provider).
    '''
    if top_similarity >= 0.80:
        return "rag_only", {"temperature": 0.1, "top_p": 0.7, "top_k": 10}
    elif top_similarity >= 0.60:
        return "rag_synthesis", {"temperature": 0.3, "top_p": 0.7, "top_k": 10}
    elif top_similarity >= 0.50:
        return "rag_hedged", {"temperature": 0.7, "top_p": 0.7, "top_k": 10}
    else:
        return "llm_only", {"temperature": 0.7, "top_p": 0.9, "top_k": 40}


# A chunk pulled into the fused top-k isn't necessarily one the model actually leaned on -- RRF
# can include a sparse-only match with no dense score at all (cosine_similarity=None), or a weak
# dense hit, purely on lexical overlap. Only cite sources with an individually solid cosine.
SOURCE_LINK_MIN_COSINE = 0.60
MAX_SOURCE_LINKS = 10


def confident_sources(results: list, min_cosine: float = SOURCE_LINK_MIN_COSINE,
                       max_links: int = MAX_SOURCE_LINKS) -> list[tuple[str, str]]:
    '''
    Which sources are worth citing: individually high cosine similarity, not just "present in
    the fused top-k". Deduplicated by (title, url) -- a page can contribute several chunks, each
    keeps only its best cosine for ranking -- and capped at max_links.
    '''
    best_cosine_by_source = {}
    for r in results:
        cos = r["cosine_similarity"]
        if cos is None or cos <= min_cosine:
            continue
        key = (r["source_title"], r["source_url"])
        if key not in best_cosine_by_source or cos > best_cosine_by_source[key]:
            best_cosine_by_source[key] = cos
    ranked = sorted(best_cosine_by_source.items(), key=lambda kv: kv[1], reverse=True)
    return [key for key, _cos in ranked[:max_links]]


# --- Small-to-big expansion: how many top results get extra page context ADDED around them ---
EXPAND_TOP_N_TO_FULL_PAGE = 1
MAX_EXPANDED_CHARS = 3000  # cap per expanded page, so one long page can't crowd out everything else

RAG_SYSTEM_PROMPT = '''You are a helpful technical assistant for a research computing center.
Answer ONLY using the information in the provided context chunks. If the context does not
contain the answer, say so explicitly instead of guessing. Always mention which source(s)
(by title) you used. Keep answers concise and technically precise. Respond in the same
language as the user's question.'''

LLM_ONLY_SYSTEM_PROMPT = '''You are a helpful technical assistant. No relevant passage was found
in the lab's documentation for this question, so answer from your general knowledge instead.
You MUST start your answer with an explicit note that this is general knowledge, not verified
against the lab's own documentation, and that the user should double-check anything specific to
their cluster/site. Respond in the same language as the user's question.'''


def build_context_block(results: list, expand_top_n: int = EXPAND_TOP_N_TO_FULL_PAGE) -> str:
    blocks = []
    for i, r in enumerate(results, 1):
        label = f"{r['source_title']} — {r['heading_path']}" if r.get("heading_path") else r["source_title"]
        if i <= expand_top_n and r["source_url"] in page_full_text_by_url:
            # Small-to-big: ADD nearby page content AROUND the matched chunk, never replace it --
            # this guarantees the exact evidence that got this chunk retrieved is always present,
            # even if it doesn't happen to fall within the first MAX_EXPANDED_CHARS of the page.
            full_page = page_full_text_by_url[r["source_url"]]
            anchor = full_page.find(r["text"][:80])  # locate roughly where the chunk sits
            if anchor == -1:
                window = full_page[:MAX_EXPANDED_CHARS]
            else:
                half = MAX_EXPANDED_CHARS // 2
                start = max(0, anchor - half)
                window = full_page[start:start + MAX_EXPANDED_CHARS]
            body = window if r["text"][:80] in window else f"{r['text']}\n\n[...page excerpt...]\n{window}"
            blocks.append(f"[Source {i}: {label} ({r['source_url']}) -- expanded excerpt]\n{body}")
        else:
            blocks.append(f"[Source {i}: {label} ({r['source_url']})]\n{r['text']}")
    return "\n\n".join(blocks)


def generate_answer(query: str, results: list, sampling_options: dict) -> str:
    context = build_context_block(results)
    user_prompt = f'''Context:
{context}

Question: {query}

Answer the question using only the context above, and cite the source title(s) you used.'''

    response = ollama.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        options=sampling_options,
    )
    return response["message"]["content"]


def generate_answer_llm_only(query: str, sampling_options: dict) -> str:
    response = ollama.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": LLM_ONLY_SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ],
        options=sampling_options,
    )
    return response["message"]["content"]

print("Generation functions ready (4 confidence bands, small-to-big expansion top",
      EXPAND_TOP_N_TO_FULL_PAGE, "result(s)).")

In [ ]:
def rag_chat(query: str, top_k_each: int = 10, top_k_final: int = 5, verbose: bool = True):
    '''
    The full conversational pipeline:
      1. Hybrid-retrieve candidates for the query.
      2. Map the best cosine similarity to a confidence band (7.1) -> a mode + sampling options.
      3. Generate accordingly: grounded-and-cited, grounded-but-hedged, or (lowest band)
         general-knowledge-and-clearly-flagged.
    '''
    results = hybrid_retrieve(query, top_k_each=top_k_each, top_k_final=top_k_final)

    # Best cosine similarity among fused results (ignore None from sparse-only hits)
    cosine_scores = [r["cosine_similarity"] for r in results if r["cosine_similarity"] is not None]
    top_similarity = max(cosine_scores) if cosine_scores else 0.0

    mode, sampling_options = sampling_for_similarity(top_similarity)

    if verbose:
        print(f"[debug] top cosine similarity = {top_similarity:.4f}  ->  mode = {mode}  "
              f"sampling = {sampling_options}")

    if mode == "llm_only":
        answer = generate_answer_llm_only(query, sampling_options)
        return (
            "🔎 Nothing in the documentation clearly matched this question, so this answer "
            "comes from the model's general knowledge, NOT your lab's documentation:\n\n"
            + answer
        )

    answer = generate_answer(query, results, sampling_options)

    if mode == "rag_hedged":
        answer = (
            "⚠️ I found some possibly related information, but I'm not fully confident "
            "it answers your exact question. Please double-check against the sources below.\n\n"
            + answer
        )

    sources = confident_sources(results)
    if not sources:
        return answer
    sources_block = "\n".join(f"  - {title} ({url})" for title, url in sources)
    return f"{answer}\n\nSources consulted:\n{sources_block}"


# Try a query that should be well covered by the corpus
print(rag_chat("How do I submit a job with SLURM?"))

In [28]:
# Try a query that is (probably) NOT covered by this corpus at all,
# to see the "llm_only" band trigger -- the answer should come back clearly
# flagged as general knowledge, not something found in your documentation.
print(rag_chat("What is the best recipe for a chocolate cake?"))

[debug] top cosine similarity = 0.3252  ->  mode = llm_only  sampling = {'temperature': 0.7, 'top_p': 0.9, 'top_k': 40}
🔎 Nothing in the documentation clearly matched this question, so this answer comes from the model's general knowledge, NOT your lab's documentation:

**Please note that the answer provided is general knowledge and not verified against the lab's own documentation.**

For a classic, moist, and delicious chocolate cake, here's a tried-and-true recipe:

Ingredients:

* 2 1/4 cups (285g) all-purpose flour
* 1 1/2 cups (185g) granulated sugar
* 2 teaspoons baking powder
* 1 teaspoon salt
* 1 cup (235ml) whole milk, at room temperature
* 2 large eggs, at room temperature
* 1/2 cup (115g) unsweetened cocoa powder
* 1 teaspoon vanilla extract
* 1/4 cup (55g) unsalted butter, melted

Instructions:

1. Preheat the oven to 350°F (180°C). Grease two 9-inch (23cm) round cake pans and line the bottoms with parchment paper.
2. In a medium bowl, whisk together flour, sugar, baking pow

<a id="sec-7-4"></a>
### 7.4 Try it yourself

Run a few of your own queries below — some that you *know* should be well covered by the
documentation you scraped, and some deliberately off-topic or vague. Watch how the debug line
(`top cosine similarity = ...`) changes, and whether the behavior (confident / hedged / refused)
matches your expectations.

In [29]:
my_queries = [
    "How much storage quota do I get by default?",
    # add your own questions here, in French or English
]

for q in my_queries:
    print("=" * 80)
    print("Q:", q)
    print(rag_chat(q))
    print()

Q: How much storage quota do I get by default?
[debug] top cosine similarity = 0.6067  ->  mode = rag_synthesis  sampling = {'temperature': 0.3, 'top_p': 0.7, 'top_k': 10}
According to the context, the default storage quota for the "HOME" directory is 20 GiB.

Sources consulted:
  - Compte utilisateur — CC-IN2P3 (https://doc.cc.in2p3.fr/fr/Daily-usage/users.html)
  - Espaces de stockage — CC-IN2P3 (https://doc.cc.in2p3.fr/fr/Data-storage/storage-areas.html)
  - Stockage de masse — CC-IN2P3 (https://doc.cc.in2p3.fr/fr/Data-storage/mass-storage.html)



<a id="sec-7-5"></a>
### 7.5 Calibrating the band boundaries properly (going further)

Rather than picking the four band boundaries (0.50 / 0.60 / 0.80) by intuition, a more rigorous
approach:

1. Build a small **evaluation set**: 15–20 questions, each labeled `"answerable"` or
   `"not answerable"` from this corpus (you write these by hand, since you know the corpus).
2. For each question, record the top cosine similarity your pipeline returns.
3. Plot the similarity distribution for `"answerable"` vs `"not answerable"` questions
   (a simple histogram is enough).
4. Pick boundaries that sit in the gaps between the two distributions — this is exactly the same
   idea as choosing a decision threshold for a binary classifier, just with more than one cut
   point. If the two distributions overlap heavily rather than separating cleanly, that's a sign
   cosine alone isn't a reliable confidence signal for your corpus — see the answerability-check
   alternative mentioned in 7.1.

This is a good exercise to do together as a group at the end of this section, using the shared
pre-scraped corpus, so that everyone converges on similar (and *justified*) boundary values.

---

<a id="sec-7-6"></a>
### 7.6 (Bonus) Comparing seven retrieval strategies side by side

Time to settle the "should I have just summarized everything?" question from earlier with actual
numbers on your own corpus, instead of intuition. We build **three more retrieval strategies** on
top of what you already have, using components you've already built (no new heavy dependency):

- **Summary**: for each chunk, ask the local LLM for an information-dense summary of up to eight
  sentences (target 90–180 words, maximum 200 words), dense-embed *that* instead of the raw chunk,
  and search by cosine similarity against the summary embeddings alone.
- **Weighted sum (summary + keywords)**: your original technique, made precise. We embed the
  summary (dense) **and** a short string of the chunk's highest-weight lexical tokens (already
  computed by BGE-M3 in Section 4 — no separate keyword-extraction step needed), then combine the
  two cosine similarities with a fixed weight `alpha` — a genuine **weighted sum of two scores**,
  as opposed to RRF's rank-based fusion in Section 6.2. Both channels here are *dense* embeddings
  (of the summary, and of the keyword string) — this is not yet a real dense+sparse hybrid.
- **Summary + sparse hybrid (RRF)**: what Section 6's hybrid pipeline would look like if its
  *dense* half were the chunk's summary embedding instead of the raw chunk — fused with the real
  sparse/lexical index (6.3) via the same Reciprocal Rank Fusion as 6.2, not a weighted average.
  This isolates one specific question: does swapping "dense over raw text" for "dense over a
  contextual summary" inside an otherwise unchanged RRF hybrid help or hurt?

Alongside the four strategies you already have (naive keyword **parse**, **sparse**, **dense**,
**hybrid**), that's seven methods total. We then run a small evaluation set of questions through
all seven and measure real numbers: `Precision@1`, `Precision@3`, `MRR`, and per-query latency.

> ⚠️ **Cost warning, revised**: the summary step calls the local LLM once per chunk, and in
> practice this runs closer to 20-45 seconds per chunk on CPU (not the 1-3s originally estimated
> here) once you account for the full ~90-180 word target response length -- on a full-site corpus
> (several hundred chunks) that's several **hours**, not minutes. `SUMMARY_MAX_CHUNKS` caps the run
> time (see 7.6.1 below) but then re-introduces the "summary methods searched a smaller subset"
> caveat discussed in 7.6.4, so the cell below caches every chunk it summarizes to
> `rag_workshop/corpus/chunk_summaries.json` / `.npz` (checkpointed every 20 new chunks) and skips
> anything already cached -- re-running it, or sharing those two files with someone else working
> from the same corpus, never repeats work already done. For a full-size corpus, prefer running
> `python3 rag_workshop/summarize_corpus.py` from a terminal instead of this cell: same cache,
> but it won't tie up this kernel, can be safely `Ctrl+C`'d and resumed, and periodically unloads
> the Ollama model (`keep_alive=0` every 50 chunks) so its memory footprint doesn't keep growing
> across hundreds of sequential calls until the machine starts swapping.

<a id="sec-7-6-1"></a>
### 7.6.1 Building the three new indexes

The next cell builds `summary_dense` and `keyword_dense` (the two new dense indexes) plus
`summary_chunk_ids` (the ID ordering they're aligned to) -- the third being the sparse index
you already have from Section 6.3, reused as-is. Skip straight to 7.6.2 if you just want the
search functions; come back here when you're ready to actually run it.

In [ ]:
import json
import os
import time

SUMMARY_MAX_CHUNKS = None        # None = summarize the WHOLE corpus (fine for this workshop's
                                  # small corpus, and needed for a fair comparison against
                                  # sparse/dense/hybrid below).
                                  # Set an int to cap the run time on a much larger corpus.
KEYWORDS_PER_CHUNK = 8           # how many top lexical tokens represent "the original keywords"
ALPHA_SUMMARY_WEIGHT = 0.6       # weighted-sum mix: alpha * summary_score + (1 - alpha) * keyword_score
SUMMARY_INPUT_MAX_CHARS = 4000    # keep enough context for conditions, examples, and exceptions

# Checkpointing / memory control -- see rag_workshop/summarize_corpus.py for the standalone,
# out-of-kernel version of this same cell (recommended for a full-size corpus, see 7.6 above).
CHECKPOINT_EVERY = 20     # save the cache to disk every N newly-computed chunks, not just at the
                          # end -- so an interrupted/killed run loses at most this many chunks.
UNLOAD_MODEL_EVERY = 50   # every N newly-computed chunks, tag that Ollama call with keep_alive=0
                          # so the server unloads the model right after responding, instead of
                          # accumulating context/state across hundreds of calls uninterrupted --
                          # this is what can make RAM climb over a long run and the machine swap.

# Cache paths -- see the note in 7.6 above. Keyed by chunk_id, so the cache can be reused across
# runs (and shared between participants) as long as the corpus was chunked the same way; a chunk
# whose ID isn't in the cache yet is simply computed and added.
SUMMARY_CACHE_JSON = os.path.join(CORPUS_DIR, "chunk_summaries.json")
SUMMARY_CACHE_NPZ = os.path.join(CORPUS_DIR, "chunk_summaries_embeddings.npz")

SUMMARY_PROMPT = '''You are an expert technical-documentation editor and information-retrieval
engineer. Create a faithful, information-dense summary of the passage below for semantic search.
The summary is a retrieval aid, not a replacement for the source: preserve the meaning and do not
invent, generalize, or add facts that are not supported by the passage.

Requirements:
- Write 4 to 8 complete sentences when the passage is substantial; use fewer only when it is truly short.
- Target 90-180 words and never exceed 200 words.
- Preserve the richest semantic context: the topic and scope, the main entities, the user's goal,
  actions and procedures, inputs and outputs, dependencies, conditions, prerequisites, constraints,
  defaults, thresholds, trade-offs, exceptions, warnings, and cause/effect relationships.
- Keep exact technical identifiers verbatim, including commands, flags, parameter names, environment
  variables, API routes, file names, package names, model names, error codes, numeric values, and
  units. Put such identifiers in backticks when natural; never paraphrase them away.
- Prefer concrete, answer-bearing facts and distinctive terminology over generic statements such as
  "this section explains". Preserve the terminology a user would likely use in a question.
- If the passage is a fragment, use its heading or local context to make the summary coherent, but
  do not infer missing details.
- Output only the summary, with no title, bullet list, commentary, or claim of certainty.

Passage:
{chunk_text}

Information-dense summary (4-8 sentences, 90-180 words, maximum 200 words):'''

def summarize_chunk(chunk_text: str, keep_alive=None) -> str:
    response = ollama.chat(
        model=GENERATION_MODEL,
        messages=[{"role": "user", "content": SUMMARY_PROMPT.format(
            chunk_text=chunk_text[:SUMMARY_INPUT_MAX_CHARS]
        )}],
        options={"temperature": 0.1, "top_p": 0.8, "top_k": 20},
        keep_alive=keep_alive,
    )
    return response["message"]["content"].strip()

def top_keywords_from_sparse(chunk_sparse_weights: dict, k: int = KEYWORDS_PER_CHUNK) -> str:
    top_tokens = sorted(chunk_sparse_weights.items(), key=lambda kv: kv[1], reverse=True)[:k]
    words = [bge_model.tokenizer.decode([int(tid)]).strip() for tid, _ in top_tokens]
    return " ".join(w for w in words if w)


def _load_summary_cache():
    '''Returns (records, vectors): records is chunk_id -> metadata dict (the JSON file, human
    -readable and diffable); vectors is chunk_id -> (summary_dense, keyword_dense) (the NPZ file,
    since embeddings don't belong in JSON). Missing/partial/corrupted cache files just mean a cold
    start (a truncated file from an interrupted previous run shouldn't crash this cell).'''
    if not (os.path.exists(SUMMARY_CACHE_JSON) and os.path.exists(SUMMARY_CACHE_NPZ)):
        return {}, {}
    try:
        with open(SUMMARY_CACHE_JSON, encoding="utf-8") as f:
            records = json.load(f)
        npz = np.load(SUMMARY_CACHE_NPZ)
        vectors = {
            chunk_id: (npz["summary_dense"][i], npz["keyword_dense"][i])
            for i, chunk_id in enumerate(npz["chunk_ids"])
        }
    except Exception as e:
        print(f"Found a summary cache, but it's corrupted/unreadable ({e}) -- starting fresh.")
        return {}, {}
    # A record with no matching vector (or vice versa) can happen if a previous run crashed
    # between writing the JSON and the NPZ -- keep only chunk_ids present in both.
    common_ids = set(records) & set(vectors)
    if len(common_ids) != len(records) or len(common_ids) != len(vectors):
        print(
            f"Summary cache has {len(records)} records but {len(vectors)} vectors -- keeping only "
            f"the {len(common_ids)} chunk_ids present in both."
        )
    records = {k: v for k, v in records.items() if k in common_ids}
    vectors = {k: v for k, v in vectors.items() if k in common_ids}
    return records, vectors


def _save_summary_cache(records, vectors):
    '''Writes through a temp file + os.replace so a crash mid-write never leaves a half-written,
    unreadable cache file behind -- the previous version stays intact until the new one is whole.'''
    tmp_json = SUMMARY_CACHE_JSON + ".tmp"
    with open(tmp_json, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    os.replace(tmp_json, SUMMARY_CACHE_JSON)

    chunk_ids = list(vectors.keys())
    tmp_npz = SUMMARY_CACHE_NPZ + ".tmp.npz"
    np.savez_compressed(
        tmp_npz,
        chunk_ids=np.array(chunk_ids),
        summary_dense=np.array([vectors[cid][0] for cid in chunk_ids]),
        keyword_dense=np.array([vectors[cid][1] for cid in chunk_ids]),
    )
    os.replace(tmp_npz, SUMMARY_CACHE_NPZ)


summary_subset = all_chunks[:SUMMARY_MAX_CHUNKS]  # SUMMARY_MAX_CHUNKS=None -> the full corpus
summary_scope_label = "full corpus" if SUMMARY_MAX_CHUNKS is None else f"top {SUMMARY_MAX_CHUNKS} chunks"

cached_records, cached_vectors = _load_summary_cache()
n_cached = sum(1 for c in summary_subset if c.chunk_id in cached_vectors)
print(f"Summarizing {len(summary_subset)} of {len(all_chunks)} chunks ({summary_scope_label})... "
      f"{n_cached} already in {os.path.basename(SUMMARY_CACHE_JSON)}.")

summary_texts, keyword_texts, summary_vecs, keyword_vecs = [], [], [], []
newly_computed = 0
interrupted = False
try:
    for c in tqdm(summary_subset, desc="Summarizing + extracting keywords"):
        if c.chunk_id in cached_vectors:
            record = cached_records[c.chunk_id]
            summary_text, keyword_text = record["summary"], record["keywords"]
            summary_vec, keyword_vec = cached_vectors[c.chunk_id]
        else:
            force_unload = UNLOAD_MODEL_EVERY and (newly_computed + 1) % UNLOAD_MODEL_EVERY == 0
            summary_text = f"{c.source_title} — {c.heading_path}\n" + summarize_chunk(
                c.text, keep_alive=0 if force_unload else None
            )
            keyword_text = top_keywords_from_sparse(sparse_index[c.chunk_id])
            encoded = bge_model.encode([summary_text, keyword_text], return_dense=True)["dense_vecs"]
            summary_vec, keyword_vec = encoded[0], encoded[1]

            cached_records[c.chunk_id] = {
                "chunk_id": c.chunk_id,
                "source_url": c.source_url,
                "source_title": c.source_title,
                "heading_path": c.heading_path,
                "text": c.text,
                "summary": summary_text,
                "keywords": keyword_text,
            }
            cached_vectors[c.chunk_id] = (summary_vec, keyword_vec)
            newly_computed += 1

            if CHECKPOINT_EVERY and newly_computed % CHECKPOINT_EVERY == 0:
                _save_summary_cache(cached_records, cached_vectors)
                tqdm.write(f"Checkpoint: {newly_computed} new chunks saved.")

        summary_texts.append(summary_text)
        keyword_texts.append(keyword_text)
        summary_vecs.append(summary_vec)
        keyword_vecs.append(keyword_vec)
except KeyboardInterrupt:
    interrupted = True
    print("\nInterrupted -- saving progress before stopping.")

_save_summary_cache(cached_records, cached_vectors)

if interrupted:
    print(
        f"Stopped after {newly_computed} new chunks this run ({len(cached_records)} total cached). "
        "Re-run this cell to resume -- already-cached chunks won't be recomputed. Consider "
        "rag_workshop/summarize_corpus.py instead for a full-size corpus (see 7.6 above): it runs "
        "outside this kernel, so it won't tie it up and can be safely left running in a terminal."
    )
else:
    if newly_computed:
        print(f"Computed {newly_computed} new ({len(summary_subset) - newly_computed} were cached); "
              f"cache now has {len(cached_records)} chunks -> {SUMMARY_CACHE_JSON}")
    else:
        print(f"All {len(summary_subset)} chunks loaded from cache -- nothing recomputed.")

    summary_dense = np.array(summary_vecs)
    keyword_dense = np.array(keyword_vecs)
    summary_chunk_ids = [c.chunk_id for c in summary_subset]

    example_summary = summary_texts[0]
    print("Example summary  :", example_summary[:500])
    print("Example word count:", len(example_summary.split()))
    print("Example keywords :", keyword_texts[0])

<a id="sec-7-6-2"></a>
### 7.6.2 Search functions for the three new strategies

`summary_search` and `weighted_sum_search` rank purely by cosine similarity over
`summary_subset` (dense-only, or a dense+dense weighted average). `summary_hybrid_search` instead
reuses `sparse_search` and `reciprocal_rank_fusion` from Section 6 as-is, only swapping in
`summary_dense` where Section 6 used the raw-chunk dense embeddings — so it's a genuine
dense+sparse RRF hybrid, just like `hybrid_retrieve`, but with a different dense half.

With `SUMMARY_MAX_CHUNKS = None` (the default set in 7.6.1), `summary_subset` **is** the full
corpus, so all three new strategies search exactly the same candidate pool as
`sparse`/`dense`/`hybrid` below — a fair, apples-to-apples comparison. If you later cap
`SUMMARY_MAX_CHUNKS` for a larger corpus, remember that `summary_subset` then only covers part of
the corpus, and the three summary-based strategies are handicapped accordingly (see 7.6.4).


In [31]:
def parse_search(query: str, chunks: list, top_k: int = 3):
    '''
    The naive floor baseline: no embeddings, no ML at all -- just count how many of the
    query's words appear as substrings in each chunk, case-insensitively.
    '''
    query_words = [w for w in query.lower().split() if len(w) > 2]
    scored = []
    for c in chunks:
        text_lower = c.text.lower()
        score = sum(text_lower.count(w) for w in query_words)
        scored.append((c.chunk_id, float(score)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


def summary_search(query_dense, top_k: int = 3):
    sims = summary_dense @ query_dense / (
        np.linalg.norm(summary_dense, axis=1) * np.linalg.norm(query_dense) + 1e-8
    )
    order = np.argsort(sims)[::-1][:top_k]
    return [(summary_chunk_ids[i], float(sims[i])) for i in order]


def weighted_sum_search(query_dense, top_k: int = 3, alpha: float = ALPHA_SUMMARY_WEIGHT):
    summary_sims = summary_dense @ query_dense / (
        np.linalg.norm(summary_dense, axis=1) * np.linalg.norm(query_dense) + 1e-8
    )
    keyword_sims = keyword_dense @ query_dense / (
        np.linalg.norm(keyword_dense, axis=1) * np.linalg.norm(query_dense) + 1e-8
    )
    combined = alpha * summary_sims + (1 - alpha) * keyword_sims
    order = np.argsort(combined)[::-1][:top_k]
    return [(summary_chunk_ids[i], float(combined[i])) for i in order]


def summary_hybrid_search(query: str, top_k_each: int = 10, top_k_final: int = 3):
    '''
    A real dense+sparse RRF hybrid where the DENSE half is the chunk's LLM-generated summary
    embedding (7.6.1) instead of the raw-chunk embedding used by Section 6's `hybrid_retrieve`.
    The SPARSE half is the exact same `sparse_search` over the exact same `sparse_index` as
    Section 6 -- nothing about the lexical side changes. Fusion is the exact same
    `reciprocal_rank_fusion` (6.2), not the weighted-average shortcut used by
    `weighted_sum_search` above.

    This answers a narrower, more useful question than "is summary search good?": *given an
    otherwise unchanged RRF hybrid pipeline, does swapping raw-chunk embeddings for summary
    embeddings on the dense side help or hurt?*
    '''
    query_encoded = bge_model.encode([query], return_dense=True, return_sparse=True)
    query_dense = query_encoded["dense_vecs"][0]
    query_sparse = query_encoded["lexical_weights"][0]

    summary_sims = summary_dense @ query_dense / (
        np.linalg.norm(summary_dense, axis=1) * np.linalg.norm(query_dense) + 1e-8
    )
    summary_order = np.argsort(summary_sims)[::-1][:top_k_each]
    summary_ranked = [(summary_chunk_ids[i], float(summary_sims[i])) for i in summary_order]

    sparse_ranked = sparse_search(query_sparse, sparse_index, top_k=top_k_each)

    fused = reciprocal_rank_fusion([summary_ranked, sparse_ranked], k=60)
    return fused[:top_k_final]


print("parse_search, summary_search, weighted_sum_search, summary_hybrid_search ready.")


parse_search, summary_search, weighted_sum_search, summary_hybrid_search ready.


<a id="sec-7-6-3"></a>
### 7.6.3 Evaluation harness

Real numbers need real (query, expected-answer-substring) pairs from **your** corpus — the ones
below are written for the CC-IN2P3 demo corpus; replace them with 5-10 pairs from whatever you
scraped. `expected_substring` is a rough relevance proxy: a lowercase snippet that should appear
in a chunk if (and pretty much only if) that chunk actually answers the question. It's imprecise
(a chunk could contain the substring by coincidence) but good enough to compare methods at a
glance, and it's the same trick you'll want for the calibration exercise in 7.5.

### 7.6.3a How to read P@1, P@3, and MRR

These metrics evaluate **retrieval**, not the quality of the final LLM answer. For each question, we
check whether a returned chunk contains the expected answer substring.

#### Standard definitions

For a ranked list of results, let `rel(i)` be 1 when the result at rank $i$ is relevant and 0
otherwise:

- **Precision@1 (P@1)**: the proportion of the first result that is relevant.
  $$P@1 = \frac{\#\text{relevant results in the first 1}}{1}$$
- **Precision@3 (P@3)**: the proportion of the first three results that are relevant.
  $$P@3 = \frac{\#\text{relevant results in the first 3}}{3}$$
- **Mean Reciprocal Rank (MRR)**: the average of the reciprocal rank of the **first** relevant result.
  $$MRR = \frac{1}{N}\sum_{q=1}^{N}\frac{1}{rank_q}$$
  where $rank_q=0$ is treated as a contribution of 0 when no relevant result is retrieved.

P@1 rewards putting the answer first. P@3 tolerates a small amount of ordering noise. MRR cares
mostly about how early the first useful result appears and penalizes questions for which no useful
result is found.

#### Important nuance in this notebook

The evaluation set stores one `expected_substring` per question, and the code records only the rank
of the **first matching chunk**. Therefore the fields named `precision@1` and `precision@3` in the
output are operationally **Hit@1** and **Hit@3**:

```text
Hit@k = proportion of questions whose first relevant chunk appears at rank <= k
```

They are useful for comparing retrieval strategies, but they are not strict textbook precision at
$k$, because the code does not label every returned chunk as relevant or irrelevant. For example,
if the first relevant ranks are `[1, 3, not found]`, this notebook reports `Hit@1 = 1/3`,
`Hit@3 = 2/3`, and `MRR = (1 + 1/3 + 0)/3 = 0.444`. A strict P@3 would require relevance labels
for all three returned chunks for every question.

> **Teaching takeaway:** use the current metrics as answer-finding metrics. If you need true P@k,
> annotate a set of relevant chunk IDs per question and count relevant results at every returned rank.

In [32]:
EVAL_QUESTIONS = [
    ("How do I submit a job with SLURM?", "sbatch"),
    ("How do I check my storage quota?", "quota"),
    ("How do I request more memory for a job?", "mem"),
    ("How do I cancel a running job?", "scancel"),
    ("What partitions are available?", "partition"),
    # Add 3-5 more pairs specific to your own corpus here.
]

chunk_lookup = {c.chunk_id: c for c in all_chunks}  # global lookup, reused by evaluate_method below

def evaluate_method(name, search_fn, top_k=3):
    hits_at_1, hits_at_3, reciprocal_ranks, latencies = 0, 0, [], []
    for question, expected_substring in EVAL_QUESTIONS:
        t0 = time.perf_counter()
        results = search_fn(question, top_k=top_k)
        latencies.append(time.perf_counter() - t0)

        found_rank = None
        for rank, (chunk_id, _score) in enumerate(results, 1):
            if expected_substring.lower() in chunk_lookup[chunk_id].text.lower():
                found_rank = rank
                break

        if found_rank == 1:
            hits_at_1 += 1
        if found_rank is not None and found_rank <= 3:
            hits_at_3 += 1
        reciprocal_ranks.append(1.0 / found_rank if found_rank else 0.0)

    n = len(EVAL_QUESTIONS)
    return {
        "method": name,
        "precision@1": hits_at_1 / n,
        "precision@3": hits_at_3 / n,
        "mrr": sum(reciprocal_ranks) / n,
        "avg_latency_ms": 1000 * sum(latencies) / n,
    }


def _dense_wrapper(query, top_k=3):
    q = bge_model.encode([query], return_dense=True)["dense_vecs"][0]
    return dense_search(q, collection, top_k=top_k)

def _sparse_wrapper(query, top_k=3):
    q = bge_model.encode([query], return_sparse=True)["lexical_weights"][0]
    return sparse_search(q, sparse_index, top_k=top_k)

def _hybrid_wrapper(query, top_k=3):
    results = hybrid_retrieve(query, top_k_each=10, top_k_final=top_k)
    return [(r["chunk_id"] if "chunk_id" in r else None, r["rrf_score"]) for r in results]

def _parse_wrapper(query, top_k=3):
    return parse_search(query, all_chunks, top_k=top_k)

def _summary_wrapper(query, top_k=3):
    q = bge_model.encode([query], return_dense=True)["dense_vecs"][0]
    return summary_search(q, top_k=top_k)

def _weighted_wrapper(query, top_k=3):
    q = bge_model.encode([query], return_dense=True)["dense_vecs"][0]
    return weighted_sum_search(q, top_k=top_k)

def _summary_hybrid_wrapper(query, top_k=3):
    return summary_hybrid_search(query, top_k_each=10, top_k_final=top_k)

rows = [
    evaluate_method("Parse (keyword count, no ML)", _parse_wrapper),
    evaluate_method("Sparse (BGE-M3 lexical)", _sparse_wrapper),
    evaluate_method("Dense (BGE-M3, raw chunk)", _dense_wrapper),
    evaluate_method("Hybrid (RRF dense+sparse, raw chunk)", _hybrid_wrapper),
    evaluate_method(f"Summary only (dense, {summary_scope_label})", _summary_wrapper),
    evaluate_method(f"Weighted sum: summary+keywords, alpha={ALPHA_SUMMARY_WEIGHT}", _weighted_wrapper),
    evaluate_method(f"Summary+sparse hybrid (RRF, {summary_scope_label})", _summary_hybrid_wrapper),
]

header = f"{'Method':<42} {'P@1':>6} {'P@3':>6} {'MRR':>6} {'Latency (ms)':>14}"
print(header)
print("-" * len(header))
for r in rows:
    print(f"{r['method']:<42} {r['precision@1']:.2f}   {r['precision@3']:.2f}   "
          f"{r['mrr']:.2f}   {r['avg_latency_ms']:>10.1f}")


Method                                        P@1    P@3    MRR   Latency (ms)
------------------------------------------------------------------------------
Parse (keyword count, no ML)               0.20   0.40   0.30          1.4
Sparse (BGE-M3 lexical)                    0.40   0.40   0.40        674.0
Dense (BGE-M3, raw chunk)                  0.40   0.60   0.47        737.3
Hybrid (RRF dense+sparse, raw chunk)       0.40   0.40   0.40       1019.9
Summary only (dense, full corpus)          0.20   0.20   0.20        668.3
Weighted sum: summary+keywords, alpha=0.6  0.20   0.60   0.37        774.5
Summary+sparse hybrid (RRF, full corpus)   0.40   0.40   0.40        792.7


<a id="sec-7-6-4"></a>
### 7.6.4 Reading the table

A few things worth noting once you have your own numbers (yours will differ from any example —
that's the point of measuring on your own corpus rather than trusting a generic benchmark):

- **Parse** is your floor: no semantics at all, purely literal substring counting. If a hybrid or
  dense method scores *worse* than parse on a question with an exact technical term in it, that's
  a red flag worth investigating (chunking cut the term awkwardly? embedding model struggling
  with a rare token?).
- With `SUMMARY_MAX_CHUNKS = None` (7.6.1's default), **summary**, **weighted sum**, and
  **summary+sparse hybrid** all search the same full corpus as sparse/dense/hybrid, so the
  comparison is fair as-is. If you capped `SUMMARY_MAX_CHUNKS` (e.g. for a much larger corpus at
  home), the three summary-based methods only searched that subset — a real handicap; don't
  conclude "summaries don't help" from a worse score without accounting for that (the fair
  comparison is then restricting the *other* methods to the same subset too).
- **Summary vs. Summary+sparse hybrid** isolates the effect of adding a real lexical channel back
  in. If `summary+sparse hybrid` beats plain `summary` (especially on exact-term questions like
  `sbatch` or `scancel`), that's evidence the summarization step is paraphrasing away exact
  technical terms that the sparse channel then recovers — the same failure mode `weighted sum`
  was designed to address, but through RRF rather than a weighted average of two dense scores.
- **Summary+sparse hybrid vs. the raw-chunk Hybrid (Section 6)** is the most direct test of your
  original observation: with everything else in the RRF pipeline held constant, is a contextual
  summary a better or worse dense representation than the raw chunk? A win here is a genuine,
  measured case for contextual retrieval on your corpus — not just intuition that "summaries seem
  to help."
- **Where summary-based search tends to genuinely help**: very long, discursive pages where even
  section-based chunking (Section 3) leaves chunks that ramble across sub-topics. A one-sentence
  summary strips the rambling and searches on the gist.
- **Where it tends to genuinely hurt**: exact technical lookups (a specific flag, error code,
  variable name) — if the summarization step paraphrases away the exact term, dense search over
  the summary embedding loses precisely the signal sparse/lexical search is best at. This is
  exactly why the hybrid RRF and weighted-sum variants keep a keyword/sparse channel alongside
  the summary — check whether either recovers some of what pure summary search lost, in your own
  numbers.
- This table only measures **retrieval** quality (did we find the right chunk), not **generation**
  quality (did the LLM's final answer use it well and stay grounded) — the two are related but
  not the same thing, per the 7.1 caveat about citations not guaranteeing support.


<a id="sec-7-6-5"></a>
### 7.6.5 Push the cached summaries into their own Chroma collection

7.6.2 below queries `summary_dense`/`keyword_dense` as plain in-memory numpy arrays, which is fine
for the small comparison in 7.6.4. But to use the summary-based retriever outside this notebook
(e.g. from a Streamlit client, the same way Section 5's `collection` is), it needs to be a real,
persisted Chroma collection instead.

**This cell only ever reads `chunk_summaries.json` / `chunk_summaries_embeddings.npz` (7.6's
cache) -- it never calls `summarize_chunk` or Ollama, no matter what.** If the cache doesn't cover
every chunk yet, the collection is simply built from whatever IS cached (and this cell tells you
how many chunks were skipped); it never triggers, and never waits on, the expensive summarization
step to fill the gaps. Run 7.6's cell (inline, fine at this notebook's small scale) or
`rag_workshop/summarize_corpus.py` (full corpus, resumable, recommended -- see 7.6 above) first for
full coverage, independently of this cell.

Only `summary_dense` is used as the embedding here, matching the `summary_search` strategy from
7.6.2 -- `weighted_sum_search`'s per-query alpha mixing and `summary_hybrid_search`'s RRF fusion
combine two separate similarity scores at query time, which isn't something a single static Chroma
embedding can express; those two stay available as the in-memory functions from 7.6.2 if you want
them instead. The Chroma **document** stored per chunk is still the original raw chunk text (not
the summary) -- the summary only steers *retrieval*; generation should still ground itself in the
actual source text, not a paraphrase of it.

In [ ]:
SUMMARY_COLLECTION_NAME = "ccin2p3_docs_summaries"

# Read-only with respect to the expensive step: only ever loads what 7.6 already cached to disk.
cached_records, cached_vectors = _load_summary_cache()
cached_ids = list(cached_vectors.keys())

missing_from_cache = sum(1 for c in all_chunks if c.chunk_id not in cached_vectors)
if missing_from_cache:
    print(
        f"{missing_from_cache} of {len(all_chunks)} current chunks have no cached summary yet -- "
        "the collection below will only cover the rest. Run 7.6's cell, or "
        "rag_workshop/summarize_corpus.py, first for full coverage -- this cell never does that "
        "itself."
    )

existing = [c.name for c in chroma_client.list_collections()]
if SUMMARY_COLLECTION_NAME in existing:
    chroma_client.delete_collection(SUMMARY_COLLECTION_NAME)
summary_collection = chroma_client.create_collection(
    name=SUMMARY_COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

documents = [cached_records[cid]["text"] for cid in cached_ids]
embeddings = [cached_vectors[cid][0].tolist() for cid in cached_ids]
metadatas = [
    {
        "source_url": cached_records[cid]["source_url"],
        "source_title": cached_records[cid]["source_title"],
        "heading_path": cached_records[cid]["heading_path"],
    }
    for cid in cached_ids
]

SUMMARY_CHROMA_BATCH_SIZE = 200
for i in tqdm(range(0, len(cached_ids), SUMMARY_CHROMA_BATCH_SIZE), desc="Inserting summaries into Chroma"):
    summary_collection.add(
        ids=cached_ids[i:i + SUMMARY_CHROMA_BATCH_SIZE],
        embeddings=embeddings[i:i + SUMMARY_CHROMA_BATCH_SIZE],
        documents=documents[i:i + SUMMARY_CHROMA_BATCH_SIZE],
        metadatas=metadatas[i:i + SUMMARY_CHROMA_BATCH_SIZE],
    )

print(f"Inserted {summary_collection.count()} chunks into Chroma collection "
      f"'{SUMMARY_COLLECTION_NAME}' (embedding = summary_dense, document = original chunk text).")

---

<a id="checkpoint"></a>
## Checkpoint

At this point you have, **entirely locally, from scratch**:

- ✅ A scraper that politely collects a technical documentation corpus.
- ✅ Two chunking strategies, and an understanding of their trade-offs.
- ✅ A single embedding model (BGE-M3) producing both dense and sparse representations.
- ✅ A persistent hybrid vector store (Chroma for dense + an in-memory sparse index).
- ✅ A from-scratch Reciprocal Rank Fusion implementation combining both.
- ✅ A conversational policy driven by retrieval confidence, with confidence-scaled sampling.
- ✅ A local LLM (via Ollama) generating cited, grounded answers.
- ✅ (Bonus) A real, measured comparison of seven retrieval strategies on your own corpus —
  including a dense+sparse RRF hybrid whose dense half is a contextual summary instead of the
  raw chunk, directly testing whether contextual retrieval helps on your own data.

**Next up (afternoon session):** we take this exact pipeline and split it across machines —
the vector store on one host, the LLM on another, and only a lightweight Streamlit client on
your laptop — then wire up a secure multi-session LLM gateway, and finally build a Thunderbird
plugin that talks to this same backend to help you answer your emails.


---

<a id="final-project"></a>
## 🚀 Final project: bring your own documentation

Everything above was built against one corpus (`doc.cc.in2p3.fr`). The real test of whether you
understood the pipeline -- not just ran it -- is applying it to a **different** documentation
source, end to end, on your own.

**The assignment**: pick a documentation source relevant to your own lab or field --

- your lab's own internal documentation website, or
- a small set of scientific papers (PDFs) relevant to your work, or
- any other technical documentation site you actually care about querying.

Then repeat the pipeline against it:

1. **Acquire the content** (adapt Section 2). If it's a website, reuse `crawl()` but change
   `SEED_URL` / `ALLOWED_DOMAIN` (and `ALLOWED_PATH_PREFIX` if you only want a subtree) to point
   at your source. **If you're using PDFs (papers)**: note that `crawl()`'s
   `SKIPPED_EXTENSIONS` deliberately *skips* `.pdf` links -- correct behavior for a documentation
   *website* (avoids parsing a binary as HTML), but it means it won't fetch papers for you.
   Extract their text instead with a library like `pypdf` or `pdfplumber`
   (`uv add pypdf` or `pip install pypdf`), and build the same
   `{"url": ..., "title": ..., "elements": [...]}` structure `crawl()` produces, so everything
   downstream (Section 3 onward) works unchanged.
2. **Chunk it** (Section 3) -- `heading_aware_chunk` works as-is if your elements carry `h1`/`h2`/
   `h3`/`p`/`li` tags; a PDF extractor that doesn't preserve headings will need
   `structure_aware_chunk` (3.1) instead, or a bit of your own heading detection on top of the
   raw extracted text.
3. **Embed it** (Section 4, and optionally 7.6 for a summary-based index too) -- no changes
   needed, `bge_model.encode(...)` doesn't care what the text is about.
4. **Give it its own Chroma collection** (Section 5) -- change `COLLECTION_NAME` to something new
   (e.g. `"my_lab_docs"`). **Do not reuse `"ccin2p3_docs"`**, or Section 5's cell will delete and
   overwrite the corpus this whole notebook was built around.
5. **Retrieve and generate against it** (Sections 6-7) -- `hybrid_retrieve()` and `rag_chat()`
   read `all_chunks`, `sparse_index`, and `collection` from the notebook's own global state, not
   as function arguments. So simply re-running Sections 2-6 with your new source's inputs already
   points everything at your new corpus -- no code changes needed in Section 6 or 7 themselves.

**What to check when you're done**: ask it a handful of real questions you'd actually want
answered from this source, at different levels of specificity. Does the confidence-banding from
7.1 behave sensibly -- high confidence on well-covered questions, honest hedging or a
general-knowledge fallback on ones your corpus doesn't cover? If a question you *know* is
answerable gets a low-confidence response, that's a chunking or embedding problem to debug with
the same tools Section 6.4 and 7.5 already gave you -- not a reason to conclude RAG "doesn't
work" for your data.

**Going further**: once this works locally, the same corpus can be pushed to a *shared* Chroma
server the way notebook 2's `rebuild_corpus.py` does for `doc.cc.in2p3.fr` -- useful if you want
your labmates querying it too, not just you.